# TIL Niche Analysis v2

In [ ]:
# Autoreload extension for IPython/Jupyter
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except (ImportError, NameError):
    pass

import pandas as pd
import numpy as np
import anndata as ad
import s3fs

import scanpy as sc
import matplotlib.pyplot as plt
import openslide
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter, AllDataCenters

sc.set_figure_params(dpi=150, color_map="viridis_r")
sc.settings.verbosity = 0

from cloudpathlib import S3Path
from scipy.sparse import issparse
from joblib import Parallel, delayed
from tqdm import tqdm

from gbmcollaboration.definitions import (
    DATA_PATH_CHUV_FILES,
    DATA_PATH_UKER_FILES,
    DATA_PATH_CR_FILES,
)
from gbmcollaboration import AllDataCenters

### Load clinical data

In [ ]:
import pandas as pd
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter

# Load processed clinical tables for all centers and combine into one cohort

def load_processed_clinical(dc, center_name):
    d = dc.load_tabular(exclude_idh1_mut=False)
    clin = d["clinical"]
    key = next((k for k in clin if "processed" in str(k).lower() and "clinical" in str(k).lower()), None)
    df = clin[key].copy() if key else list(clin.values())[0].copy()
    df["center"] = center_name
    return df

ch = load_processed_clinical(CHUVDataCenter, "CHUV")
uk = load_processed_clinical(UKERDataCenter, "UKER")
cr = load_processed_clinical(CRDataCenter, "Charite")

clin_all = pd.concat([ch, uk, cr], ignore_index=True, sort=False)
clin_out = clin_all.copy()

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

clin_out.head()

### Load single cell data

In [ ]:
data_center = AllDataCenters # or "CHUV", "UKER", "CR"
adata_sc = data_center.load_singlecell()

### Load lognorm and UMAP for single cell

In [ ]:
LOAD_DIR = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis"
LAYER = "LogNormalize"
UMAP_KEY = "X_umap"

import numpy as np, pandas as pd, scipy.sparse as sp

obs_saved = pd.read_csv(f"{LOAD_DIR}/obs_names.csv")["obs"]
var_saved = pd.read_csv(f"{LOAD_DIR}/var_names.csv")["var"]
M_saved = sp.load_npz(f"{LOAD_DIR}/{LAYER}.npz")
U_saved = pd.read_csv(f"{LOAD_DIR}/umap_coords.csv", index_col=0)

obs_cur = pd.Index(adata_sc.obs_names)
var_cur = pd.Index(adata_sc.var_names)

obs_common = obs_cur.intersection(obs_saved)
var_common = var_cur.intersection(var_saved)

cur_r = obs_cur.get_indexer(obs_common)
sav_r = pd.Index(obs_saved).get_indexer(obs_common)
cur_c = var_cur.get_indexer(var_common)
sav_c = pd.Index(var_saved).get_indexer(var_common)

sub = M_saved[sav_r, :][:, sav_c].tocoo()
rows = cur_r[sub.row]
cols = cur_c[sub.col]
M_cur = sp.csr_matrix((sub.data, (rows, cols)), shape=(len(obs_cur), len(var_cur)))
adata_sc.layers[LAYER] = M_cur

U_cur = np.full((len(obs_cur), 2), np.nan, float)
U_cur[cur_r, :] = U_saved.loc[obs_common].values
adata_sc.obsm[UMAP_KEY] = U_cur

### Load cell annotation key for single cell

In [ ]:
# user vars
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
LEVEL4_KEY = "celltype_level4_scanvi"  # source labels in your adata.obs
NEW_COLS = ["celltype_level4_curated", "celltype_level3_curated", "celltype_level3_figure",	
"celltype_level2_curated", "celltype_level2_figure", "celltype_level1_curated"]

import pandas as pd
import numpy as np

# load key
key = pd.read_csv(KEY_PATH)

# basic checks
assert LEVEL4_KEY in adata_sc.obs.columns, f"Missing {LEVEL4_KEY} in adata_sc.obs"
for c in [LEVEL4_KEY] + NEW_COLS:
    assert c in key.columns or c == LEVEL4_KEY, f"Missing column in key: {c}"

# build mapping dicts from level4 -> curated levels
key_map = key.set_index(LEVEL4_KEY)[NEW_COLS].to_dict(orient="series")

# apply mappings to create new obs columns
src = adata_sc.obs[LEVEL4_KEY].astype(str)
for col in NEW_COLS:
    mapped = src.map(key_map[col])  # may introduce NaN for unknowns
    # fill unmapped
    out = mapped.fillna("Unmapped").astype(str)
    # set as categorical, with categories ordered by appearance in key plus "Unmapped" if present
    cats = pd.Index(key[col].dropna().astype(str).unique())
    if "Unmapped" in out.values and "Unmapped" not in cats:
        cats = cats.append(pd.Index(["Unmapped"]))
    adata_sc.obs[col] = pd.Categorical(out, categories=cats)

# quick report
lvl4_all = pd.Index(src.unique())
lvl4_in_key = pd.Index(key[LEVEL4_KEY].astype(str).unique())
missing = lvl4_all.difference(lvl4_in_key)
print(f"Added columns: {NEW_COLS}")
print(f"Level4 labels in adata_sc: {len(lvl4_all)}")
print(f"Level4 labels covered by key: {len(lvl4_all.intersection(lvl4_in_key))}")
print(f"Level4 labels missing from key: {len(missing)}")
if len(missing):
    print("Unmapped labels:", ", ".join(list(missing)[:20]) + (" ..." if len(missing) > 20 else ""))

### Load Visium data from all centers

In [ ]:
adata = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    list_sample_id=None,
    normalization="log_norm_count",
    radius=1,
    deconvolution_slot_name="deconv_cohort_level_4",
)

print(adata)

### Cell2Locaction loader function

In [ ]:
# Function to load one sample
def load_sample(center_name, sample_id, h5ad_file):
    """Load and process a single sample using Cell2LocationLoader approach."""
    try:
        # Load h5ad file
        if isinstance(h5ad_file, S3Path) or str(h5ad_file).startswith('s3://'):
            s3 = s3fs.S3FileSystem(anon=False)
            h5ad_path_str = str(h5ad_file).replace('s3://', '')
            with s3.open(h5ad_path_str, 'rb') as f:
                adata = ad.read_h5ad(f)
        else:
            adata = ad.read_h5ad(h5ad_file)
        
        # Extract means_cell_abundance_w_sf from obsm
        if 'means_cell_abundance_w_sf' not in adata.obsm:
            return None
        
        # Get cell type names
        celltype_names = None
        if 'mod' in adata.uns and 'factor_names' in adata.uns['mod']:
            celltype_names = adata.uns['mod']['factor_names']
        elif 'factor_names' in adata.uns:
            celltype_names = adata.uns['factor_names']
        elif 'cell_type_names' in adata.uns:
            celltype_names = adata.uns['cell_type_names']
        
        if celltype_names is None:
            n_celltypes = adata.obsm['means_cell_abundance_w_sf'].shape[1]
            celltype_names = [f'celltype_{i}' for i in range(n_celltypes)]
        
        # Add prefix to match column names
        celltype_names = [f'meanscell_abundance_w_sf_{name}' for name in celltype_names]
        
        # Get cell abundance data
        cell_abundance = adata.obsm['means_cell_abundance_w_sf']
        if issparse(cell_abundance):
            cell_abundance = cell_abundance.toarray()
        
        # Create DataFrame
        df = pd.DataFrame(cell_abundance, index=adata.obs_names, columns=celltype_names)
        df = df.fillna(0)
        
        # Add sample ID to index
        df.index = pd.MultiIndex.from_tuples(
            [(sample_id, idx) for idx in df.index],
            names=['sample', 'barcode']
        )
        
        # Compute total cells per spot
        df['total_cells'] = df.sum(axis=1)
        
        return df
    except Exception as e:
        return None

### Load CHUV C2L abundances

In [ ]:
# Load CHUV cell2location results, if it stalls interrupt the script and start again

center_name = "CHUV"
cell2loc_path = DATA_PATH_CHUV_FILES["Cell2location"]

print(f"\nProcessing {center_name}...")
base_path = S3Path(str(cell2loc_path))
sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")

samples_to_process = [
    (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
    for sample_id in sample_dirs
]

center_results = Parallel(n_jobs=-1)(
    delayed(load_sample)(center_name, sample_id, h5ad_file)
    for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
)

center_results = [df for df in center_results if df is not None]

for df in center_results:
    new_index = pd.MultiIndex.from_tuples(
        [(center_name, *idx) for idx in df.index],
        names=["center", "sample", "barcode"],
    )
    df.index = new_index
    df["center"] = center_name

df_chuv = pd.concat(center_results, axis=0)

print(f"  Loaded {df_chuv.index.get_level_values('sample').nunique()} samples from {center_name}")
print(f"  Total spots: {df_chuv.shape[0]}")

### Load UKER C2L abundances

In [ ]:
# Load UKER cell2location results, if it stalls interrupt the script and start again

center_name = "UKER"
cell2loc_path = DATA_PATH_UKER_FILES["Cell2location"]

print(f"\nProcessing {center_name}...")
base_path = S3Path(str(cell2loc_path))
sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")

samples_to_process = [
    (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
    for sample_id in sample_dirs
]

center_results = Parallel(n_jobs=-1)(
    delayed(load_sample)(center_name, sample_id, h5ad_file)
    for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
)

center_results = [df for df in center_results if df is not None]

for df in center_results:
    new_index = pd.MultiIndex.from_tuples(
        [(center_name, *idx) for idx in df.index],
        names=["center", "sample", "barcode"],
    )
    df.index = new_index
    df["center"] = center_name

df_uker = pd.concat(center_results, axis=0)

print(f"  Loaded {df_uker.index.get_level_values('sample').nunique()} samples from {center_name}")
print(f"  Total spots: {df_uker.shape[0]}")

### Load Charite C2L abundances

In [ ]:
# Load CR cell2location results, if it stalls interrupt the script and start again

center_name = "CR"
cell2loc_path = DATA_PATH_CR_FILES["Cell2location"]

print(f"\nProcessing {center_name}...")
base_path = S3Path(str(cell2loc_path))
sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")

samples_to_process = [
    (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
    for sample_id in sample_dirs
]

center_results = Parallel(n_jobs=-1)(
    delayed(load_sample)(center_name, sample_id, h5ad_file)
    for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
)

center_results = [df for df in center_results if df is not None]

for df in center_results:
    new_index = pd.MultiIndex.from_tuples(
        [(center_name, *idx) for idx in df.index],
        names=["center", "sample", "barcode"],
    )
    df.index = new_index
    df["center"] = center_name

df_cr = pd.concat(center_results, axis=0)

print(f"  Loaded {df_cr.index.get_level_values('sample').nunique()} samples from {center_name}")
print(f"  Total spots: {df_cr.shape[0]}")

### Merge C2L abundances for all centers

In [ ]:
# Build df_all from available center-level dataframes
dfs = []

if "df_chuv" in globals():
    dfs.append(df_chuv)
if "df_uker" in globals():
    dfs.append(df_uker)
if "df_cr" in globals():
    dfs.append(df_cr)

if not dfs:
    raise RuntimeError("No center dataframes (df_chuv, df_uker, df_cr) found.")

df_all = pd.concat(dfs, axis=0)

print("Combined df_all summary:")
print("  Total spots:", df_all.shape[0])
print("  Total samples:", df_all.index.get_level_values("sample").nunique())
print("  Index levels:", df_all.index.names)
print("  Example columns:", df_all.columns[:5].tolist())

### Merge C2L abundances with Visium data

In [ ]:
# Drop the 'center' level from df_all's index for alignment
if isinstance(df_all.index, pd.MultiIndex) and "center" in df_all.index.names:
    df_all_no_center = df_all.copy()
    df_all_no_center.index = df_all_no_center.index.droplevel("center")
else:
    df_all_no_center = df_all

print("df_all_no_center index names:", df_all_no_center.index.names)
print("df_all_no_center shape:", df_all_no_center.shape)

# Build (sample, barcode) keys for each obs row, in obs order
# obs_names look like: "CH_G_151a_vis_AACACTTGGCAAGGAA-1"
barcodes = pd.Index(adata.obs_names).str.split("_").str[-1]

keys = pd.MultiIndex.from_arrays(
    [adata.obs["sample"].values, barcodes.values],
    names=["sample", "barcode"],
)

print("First 3 keys:", list(keys[:3]))
print("Number of keys:", len(keys))

# Align df_all_no_center to these keys
df_all_aligned = df_all_no_center.reindex(keys)
print("df_all_aligned shape:", df_all_aligned.shape)

# Add all df_all columns into adata.obs in obs order
added_cols = df_all_no_center.columns.tolist()
for col in added_cols:
    adata.obs[col] = df_all_aligned[col].values

print("\nNumber of added columns:", len(added_cols))
print("adata.obs shape:", adata.obs.shape)

# Quick sanity check on one row
print("\nExample row with merged C2L counts:")
print(adata.obs.iloc[0][["sample"] + added_cols[:5]].to_dict())

### Load custom cell anotations from file

In [ ]:
# %%
import pandas as pd

# user vars
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

# 1) Load key
key_raw = pd.read_csv(KEY_PATH)

# 2) Basic column checks
required_cols = [LEVEL4_KEY] + NEW_COLS
missing_cols = [c for c in required_cols if c not in key_raw.columns]
if missing_cols:
    raise ValueError(f"Missing columns in key file: {missing_cols}")

# 3) Deduplicate on the level4 key and standardize to string
key_unique = key_raw.drop_duplicates(subset=[LEVEL4_KEY]).copy()
key_unique[LEVEL4_KEY] = key_unique[LEVEL4_KEY].astype(str)

# 4) Build a tidy DataFrame indexed by level4
celltype_key = key_unique.set_index(LEVEL4_KEY)[NEW_COLS]
celltype_key = celltype_key.apply(lambda s: s.astype(str))

# 5) Attach as the canonical celltype hierarchy in adata.uns
adata.uns["celltype_annotation_key"] = celltype_key

# 6) Quick coverage checks vs Visium abundance columns
lvl4_in_key = pd.Index(celltype_key.index)

obs_cols = pd.Index(adata.obs.columns)

# deconv_cohort_level_4 columns should use the raw level4 names
deconv_level4_cols = obs_cols.intersection(lvl4_in_key)

# C2L totals use "meanscell_abundance_w_sf_<level4>"
c2l_cols = [c for c in obs_cols if c.startswith("meanscell_abundance_w_sf_")]
c2l_lvl4 = pd.Index([c.replace("meanscell_abundance_w_sf_", "") for c in c2l_cols])
c2l_in_key = c2l_lvl4.intersection(lvl4_in_key)

missing_for_deconv = lvl4_in_key.difference(deconv_level4_cols)
missing_for_c2l = lvl4_in_key.difference(c2l_in_key)

print("Stored celltype_annotation_key in adata.uns['celltype_annotation_key']")
print(f"Total level4 types in key: {len(lvl4_in_key)}")
print(f"Matching deconv level4 columns in adata.obs: {len(deconv_level4_cols)}")
print(f"Matching C2L level4 types (from meanscell_abundance_w_sf_*): {len(c2l_in_key)}")

if len(missing_for_deconv):
    print(f"Level4 types in key not seen as deconv columns: {len(missing_for_deconv)}")
if len(missing_for_c2l):
    print(f"Level4 types in key not seen in C2L columns: {len(missing_for_c2l)}")

### Check for violations of PH assumptions

In [ ]:
# %% Schoenfeld residual plot for MGMT with cleaner aesthetics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter

# ------------ user controls ------------
FIGSIZE      = (8, 4)    # width, height in inches
BASE_FONTSZ  = 10        # base font size
ROLL_WINDOW  = 25        # number of events in rolling mean
DOT_SIZE     = 12        # marker size for individual residuals
LINE_WIDTH   = 2.0       # width of rolling mean line

MGMT_COL     = "MGMT promoter methylation"
TIME_COL     = "os_years"
EVENT_COL    = "os_event"
# ---------------------------------------

# 1. Build analysis frame: OS + MGMT
df = clin_out[[TIME_COL, EVENT_COL, MGMT_COL]].copy()

# keep only complete rows
df = df.dropna(subset=[TIME_COL, EVENT_COL, MGMT_COL])

# recode MGMT to binary: 1 = methylated, 0 = unmethylated
df["MGMT_binary"] = np.where(
    df[MGMT_COL] == "Methylated", 1,
    np.where(df[MGMT_COL] == "Unmethylated", 0, np.nan)
)
df = df.dropna(subset=["MGMT_binary"])
df["MGMT_binary"] = df["MGMT_binary"].astype(int)

print(f"Patients with OS and MGMT data used in plot: {df.shape[0]}")
print(df["MGMT_binary"].value_counts().rename(index={1: "Methylated", 0: "Unmethylated"}))

# restrict to the numeric columns used in the model
df_fit = df[[TIME_COL, EVENT_COL, "MGMT_binary"]].copy()

# 2. Fit Cox PH model
cph = CoxPHFitter()
cph.fit(df_fit, duration_col=TIME_COL, event_col=EVENT_COL)

print("\nCox summary for MGMT_binary:")
print(cph.summary.loc[["MGMT_binary"]])

# 3. Schoenfeld residuals for MGMT (use the same frame as used for fitting)
res_schoen = cph.compute_residuals(df_fit, kind="schoenfeld")

# lifelines returns one column per covariate
res = res_schoen["MGMT_binary"]

# event times aligned to residuals index
times = df_fit.loc[res.index, TIME_COL]

# sort by time
order = np.argsort(times.values)
times_sorted = times.values[order]
res_sorted   = res.values[order]

# rolling mean over events
if 1 < ROLL_WINDOW < len(res_sorted):
    roll = pd.Series(res_sorted).rolling(window=ROLL_WINDOW, center=True).mean().values
else:
    roll = res_sorted.copy()

# 4. Plot with cleaner aesthetics
plt.rcParams.update({
    "font.size": BASE_FONTSZ,
    "axes.labelsize": BASE_FONTSZ,
    "axes.titlesize": BASE_FONTSZ + 1,
    "xtick.labelsize": BASE_FONTSZ - 1,
    "ytick.labelsize": BASE_FONTSZ - 1,
    "legend.fontsize": BASE_FONTSZ - 1,
})

fig, ax = plt.subplots(figsize=FIGSIZE)

# scatter of individual residuals
ax.scatter(
    times_sorted,
    res_sorted,
    s=DOT_SIZE,
    alpha=0.35,
    edgecolors="none",
    label="Schoenfeld residuals"
)

# rolling mean
ax.plot(
    times_sorted,
    roll,
    linewidth=LINE_WIDTH,
    label=f"Rolling mean (window={ROLL_WINDOW} events)"
)

# reference line for PH expectation
ax.axhline(0.0, color="black", linestyle="--", linewidth=1.0, label="PH expectation (0)")

ax.set_xlabel("Time since baseline (years)")
ax.set_ylabel("Scaled Schoenfeld residuals for MGMT_binary")
ax.set_title("Time-varying effect diagnostics for MGMT")
ax.grid(True, alpha=0.15)
ax.legend(loc="upper right", frameon=False)

plt.tight_layout()
plt.show()

### Visualize cell types of different layers by absolute abundance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# --- USER CONFIG ---
SAMPLE_ID  = "CH_G_196a_vis"           # must exist in adata.obs['sample']
LEVEL_COL  = "celltype_level2_curated"  # one of the columns in adata.uns['celltype_annotation_key']
GROUP_LABEL = "T_NK"                # one of the categories in that LEVEL_COL
SPOT_SIZE  = 60
COLORMAP   = "viridis"

# 0) sanity checks
if "celltype_annotation_key" not in adata.uns:
    raise KeyError("adata.uns['celltype_annotation_key'] not found. Run the key-loading cell first.")

for col in ["sample", "imagerow", "imagecol"]:
    if col not in adata.obs.columns:
        raise KeyError(f"adata.obs['{col}'] not found.")

key = adata.uns["celltype_annotation_key"]
if LEVEL_COL not in key.columns:
    raise KeyError(f"'{LEVEL_COL}' not found in celltype_annotation_key columns.")

# 1) find C2L columns and map them to the chosen annotation level
c2l_cols = [c for c in adata.obs.columns if c.startswith("meanscell_abundance_w_sf_")]
if not c2l_cols:
    raise ValueError("No C2L columns found (expected prefix 'meanscell_abundance_w_sf_').")

lvl4_labels = pd.Index([c.replace("meanscell_abundance_w_sf_", "") for c in c2l_cols])
key_idx = key.index.astype(str)

# keep only C2L columns whose level4 is in the key
mask_in_key = lvl4_labels.isin(key_idx)
c2l_cols_used = [col for col, keep in zip(c2l_cols, mask_in_key) if keep]
lvl4_used = lvl4_labels[mask_in_key]

if not c2l_cols_used:
    raise ValueError("No C2L columns have matching level4 entries in the key.")

# map each C2L column to the chosen annotation level (e.g. level2_figure)
lvl4_to_level = key[LEVEL_COL]
groups_for_cols = lvl4_used.map(lvl4_to_level).astype(str)

# which columns belong to the requested group?
cols_for_group = [col for col, g in zip(c2l_cols_used, groups_for_cols) if g == GROUP_LABEL]

if not cols_for_group:
    unique_groups = pd.Index(groups_for_cols.unique()).dropna().tolist()
    raise ValueError(
        f"GROUP_LABEL '{GROUP_LABEL}' not found at level '{LEVEL_COL}'. "
        f"Available groups include (first 20): {unique_groups[:20]}"
    )

# 2) subset to the chosen sample
mask_sample = adata.obs["sample"] == SAMPLE_ID
if mask_sample.sum() == 0:
    raise ValueError(f"No spots found for SAMPLE_ID = '{SAMPLE_ID}'.")

obs_s = adata.obs.loc[mask_sample].copy()

# 3) compute abundance per spot for that group (sum of matching C2L columns)
values = obs_s[cols_for_group].sum(axis=1).values
x = obs_s["imagecol"].values
y = obs_s["imagerow"].values

# 4) plot, matching your existing style
fig, ax = plt.subplots(figsize=(9.5, 7))
sc = ax.scatter(
    x, y,
    c=values,
    cmap=COLORMAP,
    s=SPOT_SIZE,
    edgecolor="k",
    linewidth=0.1,
)

ax.set_title(f"{SAMPLE_ID} – {GROUP_LABEL} ({LEVEL_COL}, C2L)")
ax.set_xlabel("imagecol")
ax.set_ylabel("imagerow")
ax.set_aspect("auto")
ax.grid(False)

pad_x = (x.max() - x.min()) * 0.02
pad_y = (y.max() - y.min()) * 0.02
ax.set_xlim(x.min() - pad_x, x.max() + pad_x)
ax.set_ylim(y.max() + pad_y, y.min() - pad_y)  # flip Y + pad

# colorbar if values vary
if values.max() != values.min():
    norm = mcolors.Normalize(vmin=values.min(), vmax=values.max())
    cb = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=COLORMAP), ax=ax)
    cb.set_label(f"{GROUP_LABEL} abundance")

plt.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
plt.show()
plt.close(fig)

### Set TIL definition and minimum threshold

In [ ]:
import pandas as pd

LEVEL_COL     = "celltype_level2_curated"
TIL_GROUPS    = ["T_NK"]
TIL_THRESHOLD = 2

key = adata.uns["celltype_annotation_key"]

# level 4 types that belong to T_NK at level2_curated
lvl4_index = key.index.astype(str)
lvl_values = key[LEVEL_COL].astype(str)
til_level4 = lvl4_index[lvl_values.isin(TIL_GROUPS)]

# corresponding C2L columns
til_c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in til_level4]

# per spot TIL_count and TIL_positive
adata.obs["TIL_count"] = adata.obs[til_c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# per-sample summary (using 'sample' as the grouping key)
til_stats = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats["TIL_fraction"] = til_stats["TIL_spots"] / til_stats["total_spots"]

### Plot top 5 TIL-high and TIL-low

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# rank samples by TIL+ spots
til_stats_sorted = til_stats.sort_values("TIL_spots", ascending=False)
top_samples = til_stats_sorted.head(5).index.tolist()
bottom_samples = til_stats_sorted.tail(5).index.tolist()

samples_to_plot = top_samples + bottom_samples

n = len(samples_to_plot)
ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)

for idx, sample_id in enumerate(samples_to_plot):
    row = idx // ncols
    col = idx % ncols
    ax = axes[row, col]

    mask = adata.obs["sample"] == sample_id
    df = adata.obs.loc[mask, ["imagerow", "imagecol", "TIL_positive"]]

    # non-TIL spots
    neg = df[~df["TIL_positive"]]
    # TIL+ spots
    pos = df[df["TIL_positive"]]

    ax.scatter(
        neg["imagecol"], neg["imagerow"],
        s=20, c="cadetblue", edgecolors="none", alpha=0.7
    )
    ax.scatter(
        pos["imagecol"], pos["imagerow"],
        s=20, c="darkred", edgecolors="none", alpha=0.9
    )

    ax.set_title(sample_id)
    ax.set_xlabel("imagecol")
    ax.set_ylabel("imagerow")
    ax.grid(False)

    if not df.empty:
        pad_x = (df["imagecol"].max() - df["imagecol"].min()) * 0.02
        pad_y = (df["imagerow"].max() - df["imagerow"].min()) * 0.02
        ax.set_xlim(df["imagecol"].min() - pad_x, df["imagecol"].max() + pad_x)
        ax.set_ylim(df["imagerow"].max() + pad_y, df["imagerow"].min() - pad_y)

# hide unused axes if fewer than 8 samples
for j in range(idx + 1, nrows * ncols):
    row = j // ncols
    col = j % ncols
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()

### Distribution of TIL-high vs TIL-low patients

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# x = total TIL+ spots, y = fraction of TIL+ spots
x = til_stats["TIL_spots"]
y = til_stats["TIL_fraction"]

# quantile thresholds on total TIL+ spots
q75 = x.quantile(0.75)
q90 = x.quantile(0.90)

mask_top25 = x >= q75
mask_top10 = x >= q90

n_total = len(x)
n_top25 = mask_top25.sum()
n_top10 = mask_top10.sum()

print(f"Total samples: {n_total}")
print(f"Top 25% (>= {q75:.1f} TIL+ spots): {n_top25}")
print(f"Remaining 75%: {n_total - n_top25}")
print(f"Top 10% (>= {q90:.1f} TIL+ spots): {n_top10}")
print(f"Remaining 90%: {n_total - n_top10}")

# color by top quartile vs others
colors = np.where(mask_top25, "darkred", "cadetblue")

plt.figure(figsize=(6, 5))
plt.scatter(x, y, s=25, c=colors, alpha=0.8)
plt.axvline(q75, linestyle="--", linewidth=1)  # 75th percentile
plt.axvline(q90, linestyle=":", linewidth=1)   # 90th percentile

plt.xlabel("Number of TIL+ spots per sample")
plt.ylabel("Fraction of TIL+ spots per sample")
plt.title("TIL+ burden per sample (current TIL definition / cutoff)")
plt.tight_layout()
plt.show()

### KM curves for TIL-high vs TIL-low

In [ ]:
# %% Survival vs TIL burden (C2L T_NK, level2_curated)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ---------------- user variables ----------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]      # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2          # C2L per spot cutoff for TIL_positive

BURDEN_METRIC  = "TIL_fraction"   # "TIL_spots" or "TIL_fraction"
SPLIT_MODE     = "top_quartile"  # "top_quartile" or "top_decile"
SURV_METRIC    = "os"          # "os" or "pfs"

TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"
# ------------------------------------------------

# 1. T_NK C2L based TIL_count and TIL_positive per spot
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_level4]

adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 2. per sample TIL stats
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]

# 3. map samples to patient IDs (strip "_vis" suffix)
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

# 4. aggregate to patient level (sum over all slides of a tumour block)
til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]

# 5. merge with clinical data
til_clin = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

# 6. choose survival columns
if SURV_METRIC == "os":
    time_col = TIME_COL_OS
    event_col = EVENT_COL_OS
    surv_label = "Overall survival"
elif SURV_METRIC == "pfs":
    time_col = TIME_COL_PFS
    event_col = EVENT_COL_PFS
    surv_label = "Progression free survival"
else:
    raise ValueError("SURV_METRIC must be 'os' or 'pfs'.")

# drop rows with missing survival info
til_clin_valid = til_clin.dropna(subset=[time_col, event_col])

# 7. choose burden metric and split mode
if BURDEN_METRIC not in ["TIL_spots", "TIL_fraction"]:
    raise ValueError("BURDEN_METRIC must be 'TIL_spots' or 'TIL_fraction'.")

burden = til_clin_valid[BURDEN_METRIC]

if SPLIT_MODE == "top_quartile":
    q = 0.75
    split_desc = "top quartile"
elif SPLIT_MODE == "top_decile":
    q = 0.90
    split_desc = "top decile"
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile' or 'top_decile'.")

cutoff = burden.quantile(q)
til_clin_valid["TIL_group"] = np.where(burden >= cutoff, "TIL_high", "TIL_low")

print(f"Patients with survival data: {til_clin_valid.shape[0]}")
print(f"TIL definition: {LEVEL_COL} {TIL_GROUPS}, C2L threshold {TIL_THRESHOLD}")
print(f"Burden metric: {BURDEN_METRIC}")
print(f"Split: {split_desc} (cutoff {cutoff:.2f} on {BURDEN_METRIC})")
print(til_clin_valid["TIL_group"].value_counts())

# 8. Kaplan Meier fit and plot
kmf_high = KaplanMeierFitter()
kmf_low = KaplanMeierFitter()

mask_high = til_clin_valid["TIL_group"] == "TIL_high"
mask_low = til_clin_valid["TIL_group"] == "TIL_low"

kmf_high.fit(
    til_clin_valid.loc[mask_high, time_col],
    event_observed=til_clin_valid.loc[mask_high, event_col],
    alpha=0.1,
    label="TIL high",
)
kmf_low.fit(
    til_clin_valid.loc[mask_low, time_col],
    event_observed=til_clin_valid.loc[mask_low, event_col],
    alpha=0.1,
    label="TIL low",
)

plt.figure(figsize=(6, 5))
kmf_high.plot(ci_show=True)
kmf_low.plot(ci_show=True)
plt.xlabel(f"{surv_label} (years)")
plt.ylabel("Survival probability")
plt.title(f"{surv_label} by TIL burden ({split_desc})")
plt.tight_layout()
plt.show()

# 9. log rank test
res = logrank_test(
    til_clin_valid.loc[mask_high, time_col],
    til_clin_valid.loc[mask_low, time_col],
    event_observed_A=til_clin_valid.loc[mask_high, event_col],
    event_observed_B=til_clin_valid.loc[mask_low, event_col],
)
print(f"Log rank p value: {res.p_value:.3e}")

In [ ]:
# %% Survival vs TIL burden (C2L T_NK, level2_curated)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ---------------- user variables ----------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]      # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2          # C2L per spot cutoff for TIL_positive

BURDEN_METRIC  = "TIL_fraction"   # "TIL_spots" or "TIL_fraction"
SPLIT_MODE     = "top_quartile"  # "top_quartile" or "top_decile"
SURV_METRIC    = "pfs"          # "os" or "pfs"

TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"
# ------------------------------------------------

# 1. T_NK C2L based TIL_count and TIL_positive per spot
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_level4]

adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 2. per sample TIL stats
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]

# 3. map samples to patient IDs (strip "_vis" suffix)
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

# 4. aggregate to patient level (sum over all slides of a tumour block)
til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]

# 5. merge with clinical data
til_clin = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

# 6. choose survival columns
if SURV_METRIC == "os":
    time_col = TIME_COL_OS
    event_col = EVENT_COL_OS
    surv_label = "Overall survival"
elif SURV_METRIC == "pfs":
    time_col = TIME_COL_PFS
    event_col = EVENT_COL_PFS
    surv_label = "Progression free survival"
else:
    raise ValueError("SURV_METRIC must be 'os' or 'pfs'.")

# drop rows with missing survival info
til_clin_valid = til_clin.dropna(subset=[time_col, event_col])

# 7. choose burden metric and split mode
if BURDEN_METRIC not in ["TIL_spots", "TIL_fraction"]:
    raise ValueError("BURDEN_METRIC must be 'TIL_spots' or 'TIL_fraction'.")

burden = til_clin_valid[BURDEN_METRIC]

if SPLIT_MODE == "top_quartile":
    q = 0.75
    split_desc = "top quartile"
elif SPLIT_MODE == "top_decile":
    q = 0.90
    split_desc = "top decile"
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile' or 'top_decile'.")

cutoff = burden.quantile(q)
til_clin_valid["TIL_group"] = np.where(burden >= cutoff, "TIL_high", "TIL_low")

print(f"Patients with survival data: {til_clin_valid.shape[0]}")
print(f"TIL definition: {LEVEL_COL} {TIL_GROUPS}, C2L threshold {TIL_THRESHOLD}")
print(f"Burden metric: {BURDEN_METRIC}")
print(f"Split: {split_desc} (cutoff {cutoff:.2f} on {BURDEN_METRIC})")
print(til_clin_valid["TIL_group"].value_counts())

# 8. Kaplan Meier fit and plot
kmf_high = KaplanMeierFitter()
kmf_low = KaplanMeierFitter()

mask_high = til_clin_valid["TIL_group"] == "TIL_high"
mask_low = til_clin_valid["TIL_group"] == "TIL_low"

kmf_high.fit(
    til_clin_valid.loc[mask_high, time_col],
    event_observed=til_clin_valid.loc[mask_high, event_col],
    alpha=0.1,
    label="TIL high",
)
kmf_low.fit(
    til_clin_valid.loc[mask_low, time_col],
    event_observed=til_clin_valid.loc[mask_low, event_col],
    alpha=0.1,
    label="TIL low",
)

plt.figure(figsize=(6, 5))
kmf_high.plot(ci_show=True)
kmf_low.plot(ci_show=True)
plt.xlabel(f"{surv_label} (years)")
plt.ylabel("Survival probability")
plt.title(f"{surv_label} by TIL burden ({split_desc})")
plt.tight_layout()
plt.show()

# 9. log rank test
res = logrank_test(
    til_clin_valid.loc[mask_high, time_col],
    til_clin_valid.loc[mask_low, time_col],
    event_observed_A=til_clin_valid.loc[mask_high, event_col],
    event_observed_B=til_clin_valid.loc[mask_low, event_col],
)
print(f"Log rank p value: {res.p_value:.3e}")

### TIL-status survival split by MGMT methylation

In [ ]:
# %% Survival vs TIL burden (C2L T_NK) stratified by MGMT methylation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ---------------- user variables ----------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]      # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2           # C2L per spot cutoff for TIL_positive

BURDEN_METRIC  = "TIL_fraction"   # "TIL_spots" or "TIL_fraction"
SPLIT_MODE     = "top_quartile"   # "top_quartile" or "top_decile"
SURV_METRIC    = "os"             # "os" or "pfs"

TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"
MGMT_COL            = "MGMT promoter methylation"
# ------------------------------------------------

# 1. T_NK C2L based TIL_count and TIL_positive per spot
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [
    f"meanscell_abundance_w_sf_{ct}"
    for ct in tnk_level4
    if f"meanscell_abundance_w_sf_{ct}" in adata.obs.columns
]

adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 2. per sample TIL stats
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = (
    til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]
)

# 3. map samples to patient IDs (strip "_vis" suffix) and aggregate to patient
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = (
    til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]
)

# 4. merge with clinical data
til_clin = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

# 5. choose survival columns
if SURV_METRIC == "os":
    time_col = TIME_COL_OS
    event_col = EVENT_COL_OS
    surv_label = "Overall survival"
elif SURV_METRIC == "pfs":
    time_col = TIME_COL_PFS
    event_col = EVENT_COL_PFS
    surv_label = "Progression free survival"
else:
    raise ValueError("SURV_METRIC must be 'os' or 'pfs'.")

# 6. clean MGMT status and drop missing survival / MGMT
til_clin["MGMT_status"] = til_clin[MGMT_COL].astype(str).str.strip()
til_clin["MGMT_status"].replace({"": np.nan}, inplace=True)

til_clin_valid = til_clin.dropna(subset=[time_col, event_col, "MGMT_status"])
til_clin_valid = til_clin_valid[
    til_clin_valid["MGMT_status"].isin(["Methylated", "Unmethylated"])
]

print(f"Patients with survival and MGMT data: {til_clin_valid.shape[0]}")

if BURDEN_METRIC not in ["TIL_spots", "TIL_fraction"]:
    raise ValueError("BURDEN_METRIC must be 'TIL_spots' or 'TIL_fraction'.")

if SPLIT_MODE == "top_quartile":
    q = 0.75
    split_desc = "top quartile"
elif SPLIT_MODE == "top_decile":
    q = 0.90
    split_desc = "top decile"
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile' or 'top_decile'.")

# 7. KM within each MGMT group
kmf_high = KaplanMeierFitter()
kmf_low = KaplanMeierFitter()

mgmt_groups = ["Methylated", "Unmethylated"]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), squeeze=False)

for i, status in enumerate(mgmt_groups):
    df = til_clin_valid[til_clin_valid["MGMT_status"] == status].copy()
    if df.empty:
        print(f"\nMGMT group {status}: no patients after filtering.")
        continue

    burden = df[BURDEN_METRIC]
    cutoff = burden.quantile(q)
    df["TIL_group"] = np.where(burden >= cutoff, "TIL_high", "TIL_low")

    print(f"\nMGMT group: {status}")
    print(f"  Split: {split_desc} on {BURDEN_METRIC}, cutoff {cutoff:.3f}")
    print(df["TIL_group"].value_counts())

    mask_high = df["TIL_group"] == "TIL_high"
    mask_low = df["TIL_group"] == "TIL_low"

    ax = axes[0, i]

    kmf_high.fit(
        df.loc[mask_high, time_col],
        event_observed=df.loc[mask_high, event_col],
        alpha=0.1,
        label="TIL high",
    )
    kmf_low.fit(
        df.loc[mask_low, time_col],
        event_observed=df.loc[mask_low, event_col],
        alpha=0.1,
        label="TIL low",
    )

    kmf_high.plot(ax=ax, ci_show=True)
    kmf_low.plot(ax=ax, ci_show=True)

    ax.set_title(f"{surv_label} by TIL in {status}")
    ax.set_xlabel(f"{surv_label} (years)")
    ax.set_ylabel("Survival probability")

    # log rank test
    res = logrank_test(
        df.loc[mask_high, time_col],
        df.loc[mask_low, time_col],
        event_observed_A=df.loc[mask_high, event_col],
        event_observed_B=df.loc[mask_low, event_col],
    )
    print(f"  Log rank p value: {res.p_value:.3e}")

plt.tight_layout()
plt.show()

### Correlation of TIL-high/low with MGMT status

In [ ]:
# %% Association between TIL_high/low and MGMT methylation

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

# ------------- user variables -------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]     # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2.0          # C2L per spot cutoff for TIL_positive

BURDEN_METRIC  = "TIL_fraction"   # "TIL_spots" or "TIL_fraction"
SPLIT_MODE     = "top_quartile"   # "top_quartile" or "top_decile"

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"
MGMT_COL            = "MGMT promoter methylation"
# -----------------------------------------

# 1) compute per-spot TIL_count and TIL_positive from C2L T_NK
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [
    f"meanscell_abundance_w_sf_{ct}"
    for ct in tnk_level4
    if f"meanscell_abundance_w_sf_{ct}" in adata.obs.columns
]

adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 2) per-sample stats
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = (
    til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]
)

# 3) map samples to patient and aggregate to patient level
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = (
    til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]
)

# 4) merge with clinical and clean MGMT
til_clin = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

til_clin["MGMT_status"] = til_clin[MGMT_COL].astype(str).str.strip()
til_clin["MGMT_status"].replace({"": np.nan}, inplace=True)

til_clin_valid = til_clin[
    til_clin["MGMT_status"].isin(["Methylated", "Unmethylated"])
].copy()

# 5) define TIL_high vs TIL_low globally
if BURDEN_METRIC not in ["TIL_spots", "TIL_fraction"]:
    raise ValueError("BURDEN_METRIC must be 'TIL_spots' or 'TIL_fraction'.")

burden = til_clin_valid[BURDEN_METRIC]

if SPLIT_MODE == "top_quartile":
    q = 0.75
    split_desc = "top quartile"
elif SPLIT_MODE == "top_decile":
    q = 0.90
    split_desc = "top decile"
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile' or 'top_decile'.")

cutoff = burden.quantile(q)
til_clin_valid["TIL_group"] = np.where(burden >= cutoff, "TIL_high", "TIL_low")

print(f"Patients with MGMT and TIL data: {til_clin_valid.shape[0]}")
print(f"TIL definition: {LEVEL_COL} {TIL_GROUPS}, C2L threshold {TIL_THRESHOLD}")
print(f"Burden metric: {BURDEN_METRIC}")
print(f"Split: {split_desc} (cutoff {cutoff:.3f} on {BURDEN_METRIC})")
print(til_clin_valid["TIL_group"].value_counts())

# 6) contingency table MGMT x TIL_group
ct = pd.crosstab(til_clin_valid["MGMT_status"], til_clin_valid["TIL_group"])
print("\nContingency table (MGMT_status x TIL_group):")
print(ct)

# row percentages
row_pct = ct.div(ct.sum(axis=1), axis=0) * 100
print("\nRow percentages (% within MGMT_status):")
print(row_pct.round(1))

# column percentages
col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
print("\nColumn percentages (% within TIL_group):")
print(col_pct.round(1))

# 7) chi-square test and phi coefficient
chi2, p_chi2, dof, expected = chi2_contingency(ct)
n = ct.values.sum()
phi = np.sqrt(chi2 / n)

print(f"\nChi-square test of independence:")
print(f"  chi2 = {chi2:.3f}, df = {dof}, p = {p_chi2:.3e}")
print(f"  Phi coefficient (effect size) = {phi:.3f}")

# 8) Fisher's exact (for 2x2) and odds ratio
if ct.shape == (2, 2):
    oddsratio, p_fisher = fisher_exact(ct.values)
    print("\nFisher's exact test (2x2):")
    print(f"  Odds ratio (TIL_high vs TIL_low, Unmethylated vs Methylated) = {oddsratio:.3f}")
    print(f"  p = {p_fisher:.3e}")

### Multifactor Cox modeling with MGMT and Age (T_NK as linear variable)

In [ ]:
# %% Multifactor Cox models: T_NK with MGMT and age_at_diagnosis

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# ---------------- user variables ----------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]      # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2             # C2L per spot cutoff for TIL_positive

BURDEN_METRIC  = "TIL_fraction"   # "TIL_spots" or "TIL_fraction"
SURV_METRIC    = "os"             # "os" or "pfs"

TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

PATIENT_ID_COL_CLIN   = "Tumour block ID for MOSAIC"
MGMT_COL              = "MGMT promoter methylation"
AGE_COL               = "age_at_diagnosis"
# ------------------------------------------------

# 1. Identify T_NK-related C2L columns from celltype_annotation_key
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_level4]

# 2. Spot-level TIL burden and positivity
adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 3. Per-sample TIL stats
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]

# 4. Map samples to patient IDs (strip "_vis") and aggregate to patient level
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]

# 5. Merge with clinical data (MGMT + age + survival)
til_clin = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

if SURV_METRIC == "os":
    time_col = TIME_COL_OS
    event_col = EVENT_COL_OS
elif SURV_METRIC == "pfs":
    time_col = TIME_COL_PFS
    event_col = EVENT_COL_PFS
else:
    raise ValueError("SURV_METRIC must be 'os' or 'pfs'.")

# 6. Build Cox dataframe
df = til_clin[[time_col, event_col, BURDEN_METRIC, MGMT_COL, AGE_COL]].copy()

# survival columns
df = df.dropna(subset=[time_col, event_col, BURDEN_METRIC])

# MGMT: binary 1 = Methylated, 0 = Unmethylated, others dropped
df["MGMT_binary"] = df[MGMT_COL].map({"Methylated": 1, "Unmethylated": 0})
df = df.dropna(subset=["MGMT_binary"])

# age numeric
df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")
df = df.dropna(subset=[AGE_COL])

# z-score TIL_fraction and age so HR is per SD
df["TIL_fraction_z"] = (df[BURDEN_METRIC] - df[BURDEN_METRIC].mean()) / df[BURDEN_METRIC].std()
df["age_z"] = (df[AGE_COL] - df[AGE_COL].mean()) / df[AGE_COL].std()

print(f"Patients with complete data for Cox models: {df.shape[0]}")
print(df[["MGMT_binary"]].value_counts().rename("n_by_MGMT_binary"))
print()

# helper to run and print a Cox model
def run_cox_model(df_in, covariates, label):
    df_cox = df_in[[time_col, event_col] + covariates].copy()
    df_cox = df_cox.rename(columns={time_col: "time", event_col: "event"})
    cph = CoxPHFitter()
    cph.fit(df_cox, duration_col="time", event_col="event")
    print(f"=== {label} ===")
    print(cph.summary)
    print()
    return cph

# 7. Models
cov_TIL        = ["TIL_fraction_z"]
cov_TIL_MGMT   = ["TIL_fraction_z", "MGMT_binary"]
cov_TIL_AGE    = ["TIL_fraction_z", "age_z"]
cov_TIL_MGMT_AGE = ["TIL_fraction_z", "MGMT_binary", "age_z"]

cph_TIL       = run_cox_model(df, cov_TIL, "T_NK (TIL_fraction_z) only")
cph_TIL_MGMT  = run_cox_model(df, cov_TIL_MGMT, "T_NK + MGMT_binary")
cph_TIL_AGE   = run_cox_model(df, cov_TIL_AGE, "T_NK + age_z")
cph_full      = run_cox_model(df, cov_TIL_MGMT_AGE, "T_NK + MGMT_binary + age_z")

### Multifactor Cox modeling with MGMT and Age (T_NK as non-linear variable)

In [ ]:
# %% Multivariable Cox with non-linear T_NK (quartiles) + MGMT + age

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# ---------------- user parameters ----------------
LEVEL_COL           = "celltype_level2_curated"
TIL_GROUPS          = ["T_NK"]        # treated as TILs at level2_curated
TIL_THRESHOLD       = 2.0            # C2L cutoff for TIL_positive

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"
TIME_COL_OS         = "os_years"
EVENT_COL_OS        = "os_event"
MGMT_COL            = "MGMT promoter methylation"
AGE_COL             = "age_at_diagnosis"
# -------------------------------------------------

# 1. Identify level4 factors that belong to T_NK at level2
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_level4]

# 2. Spot-level TIL counts and positivity
adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 3. Aggregate to sample level
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = (
    til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]
)

# 4. Map samples to patient IDs (strip "_vis") and aggregate to patient level
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

til_stats_patient = til_stats_sample.groupby("patient_id", as_index=False).agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = (
    til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]
)

# 5. Merge with clinical table
df = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

# 6. Keep relevant columns and clean types
df = df[[
    "patient_id",
    "TIL_fraction",
    TIME_COL_OS,
    EVENT_COL_OS,
    MGMT_COL,
    AGE_COL,
]].copy()

# MGMT: 1 = methylated, 0 = unmethylated, drop anything else
df["MGMT_binary"] = df[MGMT_COL].map({"Unmethylated": 0, "Methylated": 1})

# Age: ensure numeric
df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")

# Drop rows missing any of the required fields
df = df.dropna(subset=[TIME_COL_OS, EVENT_COL_OS, "MGMT_binary", AGE_COL, "TIL_fraction"])

# Standardize age
df["age_z"] = (df[AGE_COL] - df[AGE_COL].mean()) / df[AGE_COL].std()

print(f"Patients with complete data: {df.shape[0]}")

# 7. Non-linear T_NK term: quartiles of TIL_fraction
q = df["TIL_fraction"].quantile([0.25, 0.50, 0.75])
bins = [-np.inf, q[0.25], q[0.50], q[0.75], np.inf]
labels = ["Q1_low", "Q2_midlow", "Q3_midhigh", "Q4_high"]

df["TIL_q"] = pd.cut(df["TIL_fraction"], bins=bins, labels=labels)
df["TIL_q"] = pd.Categorical(df["TIL_q"], categories=labels, ordered=True)

print("\nT_NK burden quartile counts:")
print(df["TIL_q"].value_counts().sort_index())

# 8. Fit Cox model with categorical T_NK term + MGMT + age
cph = CoxPHFitter()
cph.fit(
    df[[TIME_COL_OS, EVENT_COL_OS, "TIL_q", "MGMT_binary", "age_z"]],
    duration_col=TIME_COL_OS,
    event_col=EVENT_COL_OS,
    formula="C(TIL_q) + MGMT_binary + age_z",
)

print("\nMultivariable Cox with non-linear T_NK (quartiles) + MGMT + age")
print(cph.summary)

# Explicit hazard ratios (vs Q1_low for the TIL_q levels)
hr = np.exp(cph.params_)
print("\nHazard ratios (relative to Q1_low for TIL_q terms):")
print(hr)

### Multifactor RMST modeling with MGMT and Age

In [ ]:
# %% RMST by T_NK burden quartiles (OS, tau = 2 and 3 years)

import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# This cell assumes 'df' is the same DataFrame built in the Cox quartile model cell.
# Required columns: "TIL_q", "os_years", "os_event"

TIME_COL = "os_years"
EVENT_COL = "os_event"

# Make a local copy so we do not mutate the Cox frame by accident
df_rmst = df[[ "patient_id", "TIL_q", TIME_COL, EVENT_COL ]].copy()

# Drop any rows without quartile, time, or event
df_rmst = df_rmst.dropna(subset=["TIL_q", TIME_COL, EVENT_COL])

quartiles = ["Q1_low", "Q2_midlow", "Q3_midhigh", "Q4_high"]
TAUS = [2.0, 3.0]

kmf = KaplanMeierFitter()

def rmst_from_km(t, s, tau):
    t = np.asarray(t, float)
    s = np.asarray(s, float)

    order = np.argsort(t)
    t = t[order]
    s = s[order]

    # Clip times at tau
    t_clipped = np.clip(t, None, tau)

    # For any times beyond tau, hold survival at S(tau-)
    idx_tau = np.searchsorted(t, tau, side="right") - 1
    if idx_tau < 0:
        # no event before tau, RMST is tau
        return float(tau)

    s_trunc = s.copy()
    s_trunc[t > tau] = s[idx_tau]

    dt = np.diff(np.concatenate(([0.0], t_clipped)))
    return float(np.sum(dt * s_trunc))

rows_all = []
rows_contrast = []

for tau in TAUS:
    # RMST per quartile
    for cat in quartiles:
        mask = df_rmst["TIL_q"] == cat
        n = int(mask.sum())
        if n == 0:
            continue

        kmf.fit(
            df_rmst.loc[mask, TIME_COL],
            event_observed=df_rmst.loc[mask, EVENT_COL],
        )
        t = kmf.survival_function_.index.values
        s = kmf.survival_function_["KM_estimate"].values
        rmst_val = rmst_from_km(t, s, tau)

        rows_all.append({
            "TIL_q": cat,
            "tau_years": tau,
            "n": n,
            "RMST": rmst_val,
        })

    # Q4_high vs Q1_low contrast
    mask_high = df_rmst["TIL_q"] == "Q4_high"
    mask_low  = df_rmst["TIL_q"] == "Q1_low"
    n_high = int(mask_high.sum())
    n_low  = int(mask_low.sum())

    if n_high > 0 and n_low > 0:
        # RMST for Q4_high
        kmf.fit(
            df_rmst.loc[mask_high, TIME_COL],
            event_observed=df_rmst.loc[mask_high, EVENT_COL],
        )
        t_high = kmf.survival_function_.index.values
        s_high = kmf.survival_function_["KM_estimate"].values
        rmst_high = rmst_from_km(t_high, s_high, tau)

        # RMST for Q1_low
        kmf.fit(
            df_rmst.loc[mask_low, TIME_COL],
            event_observed=df_rmst.loc[mask_low, EVENT_COL],
        )
        t_low = kmf.survival_function_.index.values
        s_low = kmf.survival_function_["KM_estimate"].values
        rmst_low = rmst_from_km(t_low, s_low, tau)

        # Log rank p for Q4 vs Q1 (standard, not RMST specific, but complementary)
        lr_res = logrank_test(
            df_rmst.loc[mask_high, TIME_COL],
            df_rmst.loc[mask_low, TIME_COL],
            event_observed_A=df_rmst.loc[mask_high, EVENT_COL],
            event_observed_B=df_rmst.loc[mask_low, EVENT_COL],
        )

        rows_contrast.append({
            "tau_years": tau,
            "n_high_Q4": n_high,
            "n_low_Q1": n_low,
            "RMST_high_Q4": rmst_high,
            "RMST_low_Q1": rmst_low,
            "RMST_diff_Q4_minus_Q1": rmst_high - rmst_low,
            "logrank_p_Q4_vs_Q1": lr_res.p_value,
        })

rmst_table = pd.DataFrame(rows_all).sort_values(["tau_years", "TIL_q"])
contrast_table = pd.DataFrame(rows_contrast).sort_values("tau_years")

print("RMST by T_NK burden quartile (OS):")
print(rmst_table)

print("\nQ4_high vs Q1_low RMST contrast (OS):")
print(contrast_table)

# Optional: save to CSV
rmst_table.to_csv("RMST_TNK_quartiles_OS.csv", index=False)
contrast_table.to_csv("RMST_TNK_Q4_vs_Q1_OS.csv", index=False)

### Visium vs single cell TIL stratification

In [ ]:
# %% Visium %TIL+ vs scRNA T_NK fraction (explicit keys: Visium=sample, scRNA=orig.ident)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------- user settings -------------
VISIUM_KEY = "sample"       # column in adata.obs with Visium sample IDs
SCRNA_KEY  = "orig.ident"   # column in adata_sc.obs with matching IDs

# If Visium has CH_G_196a_vis and scRNA has CH_G_196a, keep this True
STRIP_VISIUM_SUFFIX = True
VISIUM_SUFFIX = "_vis"

LEVEL2_COL = "celltype_level2_curated"
TIL_LEVEL2 = ["T_NK"]
# ----------------------------------------

# sanity checks
if VISIUM_KEY not in adata.obs.columns:
    raise KeyError(
        f"'{VISIUM_KEY}' not in adata.obs.columns. "
        f"Available: {adata.obs.columns.tolist()}"
    )
if SCRNA_KEY not in adata_sc.obs.columns:
    raise KeyError(
        f"'{SCRNA_KEY}' not in adata_sc.obs.columns. "
        f"Available: {adata_sc.obs.columns.tolist()}"
    )
if LEVEL2_COL not in adata_sc.obs.columns:
    raise KeyError(f"'{LEVEL2_COL}' not in adata_sc.obs.columns.")

# Visium TIL flag
if "TIL_positive" in adata.obs.columns:
    til_col = "TIL_positive"
elif "TIL_plus" in adata.obs.columns:
    til_col = "TIL_plus"
else:
    raise KeyError(
        "No TIL boolean column found in adata.obs "
        "(expected 'TIL_positive' or 'TIL_plus')."
    )

# 1) Visium: % TIL+ spots per vis_id
vis_ids = adata.obs[VISIUM_KEY].astype(str)
if STRIP_VISIUM_SUFFIX and VISIUM_SUFFIX:
    vis_ids = vis_ids.str.replace(f"{VISIUM_SUFFIX}$", "", regex=True)

vis_df = pd.DataFrame(
    {
        "vis_id": vis_ids.values,
        "TIL_positive": adata.obs[til_col].astype(bool).values,
    }
)
vis_df["Total_Spots"] = 1

vis_grp = (
    vis_df.groupby("vis_id", as_index=False)
          .agg(TIL_Positive_Spots=("TIL_positive", "sum"),
               Total_Spots=("Total_Spots", "sum"))
)
vis_grp["Visium_TILP_percent"] = (
    100.0 * vis_grp["TIL_Positive_Spots"] / vis_grp["Total_Spots"]
)

# 2) scRNA: T_NK fraction per scr_id
scr_ids = adata_sc.obs[SCRNA_KEY].astype(str)
scr_df = pd.DataFrame(
    {
        "scr_id": scr_ids.values,
        LEVEL2_COL: adata_sc.obs[LEVEL2_COL].astype(str).values,
    }
).dropna(subset=[LEVEL2_COL])

scr_df["is_TNK"] = scr_df[LEVEL2_COL].isin(TIL_LEVEL2)

scr_grp = (
    scr_df.groupby("scr_id", as_index=False)
          .agg(T_NK_cells=("is_TNK", "sum"),
               total_cells=(LEVEL2_COL, "size"))
)
scr_grp["T_NK_fraction"] = scr_grp["T_NK_cells"] / scr_grp["total_cells"]

# 3) quartiles per modality (full cohorts)
q_vis = vis_grp["Visium_TILP_percent"].quantile(0.75)
q_scr = scr_grp["T_NK_fraction"].quantile(0.75)

vis_grp["vis_hi"] = vis_grp["Visium_TILP_percent"] >= q_vis
scr_grp["scr_hi"] = scr_grp["T_NK_fraction"] >= q_scr

n_vis_total = vis_grp.shape[0]
n_scr_total = scr_grp.shape[0]
n_vis_high  = int(vis_grp["vis_hi"].sum())
n_scr_high  = int(scr_grp["scr_hi"].sum())

print(f"Visium TIL-high (>= Q3={q_vis:.3f}%): {n_vis_high}/{n_vis_total}")
print(f"scRNA  T_NK-high (>= Q3={q_scr:.3f}): {n_scr_high}/{n_scr_total}")

# 4) merge for plotting and intersection counts
df = vis_grp.merge(
    scr_grp,
    left_on="vis_id",
    right_on="scr_id",
    how="inner",
)

if df.empty:
    raise RuntimeError(
        "No intersection between Visium IDs and scRNA IDs. "
        "Check VISIUM_KEY, SCRNA_KEY, and STRIP_VISIUM_SUFFIX."
    )

vis_hi = df["vis_hi"]
scr_hi = df["scr_hi"]

n_intersection = len(df)
n_both_high    = int((vis_hi & scr_hi).sum())
n_vis_only     = int((vis_hi & ~scr_hi).sum())
n_scr_only     = int((~vis_hi & scr_hi).sum())
n_neither      = int((~vis_hi & ~scr_hi).sum())

print(
    f"Intersection (n={n_intersection}): "
    f"both high={n_both_high}, "
    f"vis-only={n_vis_only}, "
    f"scr-only={n_scr_only}, "
    f"neither={n_neither}"
)

# 5) category for coloring
df["quartile_category"] = np.select(
    [vis_hi & scr_hi, vis_hi & ~scr_hi, ~vis_hi & scr_hi],
    ["both_hi", "vis_hi_only", "scr_hi_only"],
    default="neither_hi",
)

# 6) scatter plot
color_map = {
    "both_hi": "orange",
    "vis_hi_only": "blue",
    "scr_hi_only": "green",
    "neither_hi": "gray",
}

plt.figure(figsize=(7, 6))
plt.scatter(
    df["Visium_TILP_percent"],
    df["T_NK_fraction"],
    c=df["quartile_category"].map(color_map),
    s=40,
    edgecolors="black",
    linewidths=0.3,
)
plt.axvline(q_vis, linestyle="--", alpha=0.5)
plt.axhline(q_scr, linestyle="--", alpha=0.5)
plt.xlabel("Visium % TIL+ spots")
plt.ylabel("scRNA T_NK fraction")
plt.title("Visium vs scRNA with upper quartile coloring")
plt.grid(False)

for label, color in [
    ("Visium ∧ scRNA upper quartile", "orange"),
    ("Visium upper quartile only", "blue"),
    ("scRNA upper quartile only", "green"),
    ("Neither upper quartile", "gray"),
]:
    plt.scatter([], [], c=color, s=40, edgecolors="black", linewidths=0.3, label=label)

plt.legend(frameon=False, loc="best")
plt.tight_layout()
plt.show()

# 7) export and store thresholds
out = df[["vis_id", "Visium_TILP_percent", "T_NK_fraction", "quartile_category"]].copy()
out["q_vis"] = q_vis
out["q_scr"] = q_scr
out.to_csv("TIL_visium_vs_scrna_upper_quartiles.csv", index=False)

adata.uns["TIL_quartiles"] = {
    "Visium_TILP_percent_q75": float(q_vis),
    "T_NK_fraction_q75": float(q_scr),
}

In [ ]:
# %% Survival analysis using single cell T_NK fraction (snRNA) to define TIL_high vs TIL_low

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ------------- user settings -------------

# scRNA annotation
LEVEL2_COL   = "celltype_level2_curated"
TIL_LEVEL2   = ["T_NK"]          # which level2_curated labels are treated as T/NK
SCRNA_KEY    = "orig.ident"      # sample id in adata_sc.obs

# clinical mapping
CLIN_ID_COL  = "Tumour block ID for MOSAIC"  # patient id in clin_out
TIME_COL     = "os_years"       # or "pfs_years"
EVENT_COL    = "os_event"       # or "pfs_event"

# how to define TIL_high vs TIL_low
BURDEN_METRIC = "T_NK_fraction"  # do not change unless you add other metrics
SPLIT_MODE    = "top_quartile"   # "top_quartile", "top_10pct", or "threshold"
CUSTOM_CUT    = 0.027             # used only if SPLIT_MODE == "threshold"

PLOT_TITLE    = f"snRNA T_NK based survival ({BURDEN_METRIC}, {SPLIT_MODE})"

# ----------------------------------------

# 1) build per sample T_NK fraction from adata_sc
if LEVEL2_COL not in adata_sc.obs.columns:
    raise KeyError(f"'{LEVEL2_COL}' not found in adata_sc.obs")

if SCRNA_KEY not in adata_sc.obs.columns:
    raise KeyError(f"'{SCRNA_KEY}' not found in adata_sc.obs")

obs_sc = adata_sc.obs[[SCRNA_KEY, LEVEL2_COL]].copy()
obs_sc = obs_sc.dropna(subset=[LEVEL2_COL])

obs_sc["is_TNK"] = obs_sc[LEVEL2_COL].isin(TIL_LEVEL2)

scrna_burden = (
    obs_sc
    .groupby(SCRNA_KEY, as_index=False)
    .agg(
        T_NK_cells=("is_TNK", "sum"),
        total_cells=(LEVEL2_COL, "size"),
    )
)
scrna_burden["T_NK_fraction"] = scrna_burden["T_NK_cells"] / scrna_burden["total_cells"]

# 2) join with clinical data
if CLIN_ID_COL not in clin_out.columns:
    raise KeyError(f"'{CLIN_ID_COL}' not found in clin_out.columns")

if TIME_COL not in clin_out.columns:
    raise KeyError(f"'{TIME_COL}' not found in clin_out.columns")

if EVENT_COL not in clin_out.columns:
    raise KeyError(f"'{EVENT_COL}' not found in clin_out.columns")

clin_subset = clin_out[[CLIN_ID_COL, TIME_COL, EVENT_COL]].copy()

# assume orig.ident strings match Tumour block ID for MOSAIC
scrna_burden = scrna_burden.rename(columns={SCRNA_KEY: CLIN_ID_COL})

til_clin = scrna_burden.merge(clin_subset, on=CLIN_ID_COL, how="inner")

# drop rows without survival info
til_clin = til_clin.dropna(subset=[TIME_COL, EVENT_COL, BURDEN_METRIC])

# 3) define cutoff and TIL groups
if SPLIT_MODE == "top_quartile":
    cutoff = til_clin[BURDEN_METRIC].quantile(0.75)
elif SPLIT_MODE == "top_10pct":
    cutoff = til_clin[BURDEN_METRIC].quantile(0.90)
elif SPLIT_MODE == "threshold":
    cutoff = float(CUSTOM_CUT)
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile', 'top_10pct', or 'threshold'.")

til_clin["TIL_group"] = np.where(
    til_clin[BURDEN_METRIC] >= cutoff,
    "TIL_high",
    "TIL_low",
)

group_counts = til_clin["TIL_group"].value_counts()
print(f"Patients with survival and scRNA TIL data: {til_clin.shape[0]}")
print(f"TIL definition (scRNA): {LEVEL2_COL} {TIL_LEVEL2}")
print(f"Burden metric: {BURDEN_METRIC}")
print(f"Split: {SPLIT_MODE} (cutoff {cutoff:.3g} on {BURDEN_METRIC})")
print(group_counts)

# 4) Kaplan Meier and logrank test
high_mask = til_clin["TIL_group"] == "TIL_high"
low_mask  = til_clin["TIL_group"] == "TIL_low"

T_high = til_clin.loc[high_mask, TIME_COL].astype(float)
E_high = til_clin.loc[high_mask, EVENT_COL].astype(int)

T_low  = til_clin.loc[low_mask, TIME_COL].astype(float)
E_low  = til_clin.loc[low_mask, EVENT_COL].astype(int)

kmf_high = KaplanMeierFitter()
kmf_low  = KaplanMeierFitter()

plt.figure(figsize=(7, 6))

kmf_high.fit(T_high, event_observed=E_high, label="TIL_high (snRNA)")
kmf_high.plot(ci_show=True, ci_alpha=0.25)

kmf_low.fit(T_low, event_observed=E_low, label="TIL_low (snRNA)")
kmf_low.plot(ci_show=True, ci_alpha=0.25)

plt.xlabel(TIME_COL)
plt.ylabel("Survival probability")
plt.title(PLOT_TITLE)
plt.grid(False)
plt.tight_layout()

plt.show()

# 5) logrank test
lr = logrank_test(T_high, T_low, event_observed_A=E_high, event_observed_B=E_low)
print(f"Logrank test p value: {lr.p_value:.3e}")

In [ ]:
# %% Survival: patients high in BOTH Visium and snRNA TIL (top quartile in each)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ------------- user settings -------------

# ID columns
VISIUM_KEY = "sample"       # in adata.obs, e.g. "CH_G_196a_vis"
SCRNA_KEY  = "orig.ident"   # in adata_sc.obs, e.g. "CH_G_196a"

# if Visium IDs have a "_vis" suffix that snRNA and clin_out do not
STRIP_VISIUM_SUFFIX = True
VISIUM_SUFFIX = "_vis"

# T/NK definition in single-cell
LEVEL2_COL = "celltype_level2_curated"
TIL_LEVEL2 = ["T_NK"]

# clinical data
CLIN_ID_COL = "Tumour block ID for MOSAIC"
TIME_COL    = "os_years"    # or "pfs_years"
EVENT_COL   = "os_event"    # or "pfs_event"

PLOT_TITLE  = "Survival: TIL-high in both Visium and snRNA"

# -----------------------------------------
# basic checks
for col, where in [
    (VISIUM_KEY, "adata.obs"),
    (LEVEL2_COL, "adata_sc.obs"),
    (SCRNA_KEY, "adata_sc.obs"),
    (CLIN_ID_COL, "clin_out"),
    (TIME_COL, "clin_out"),
    (EVENT_COL, "clin_out"),
]:
    if where == "adata.obs" and col not in adata.obs.columns:
        raise KeyError(f"'{col}' not in adata.obs.columns")
    if where == "adata_sc.obs" and col not in adata_sc.obs.columns:
        raise KeyError(f"'{col}' not in adata_sc.obs.columns")
    if where == "clin_out" and col not in clin_out.columns:
        raise KeyError(f"'{col}' not in clin_out.columns")

# TIL flag in Visium
if "TIL_positive" in adata.obs.columns:
    til_col = "TIL_positive"
elif "TIL_plus" in adata.obs.columns:
    til_col = "TIL_plus"
else:
    raise KeyError("Expected 'TIL_positive' or 'TIL_plus' in adata.obs.")

# 1) Visium: TIL+ fraction per sample
vis_ids = adata.obs[VISIUM_KEY].astype(str)
if STRIP_VISIUM_SUFFIX and VISIUM_SUFFIX:
    vis_ids = vis_ids.str.replace(f"{VISIUM_SUFFIX}$", "", regex=True)

vis_df = pd.DataFrame(
    {
        "vis_id": vis_ids.values,
        "TIL_positive": adata.obs[til_col].astype(bool).values,
    }
)
vis_df["Total_Spots"] = 1

vis_grp = (
    vis_df.groupby("vis_id", as_index=False)
          .agg(
              TIL_Positive_Spots=("TIL_positive", "sum"),
              Total_Spots=("Total_Spots", "sum"),
          )
)
vis_grp["Visium_TILP_fraction"] = vis_grp["TIL_Positive_Spots"] / vis_grp["Total_Spots"]

# 2) snRNA: T_NK fraction per sample
scr_ids = adata_sc.obs[SCRNA_KEY].astype(str)
scr_df = pd.DataFrame(
    {
        "scr_id": scr_ids.values,
        LEVEL2_COL: adata_sc.obs[LEVEL2_COL].astype(str).values,
    }
).dropna(subset=[LEVEL2_COL])

scr_df["is_TNK"] = scr_df[LEVEL2_COL].isin(TIL_LEVEL2)

scr_grp = (
    scr_df.groupby("scr_id", as_index=False)
          .agg(
              T_NK_cells=("is_TNK", "sum"),
              total_cells=(LEVEL2_COL, "size"),
          )
)
scr_grp["T_NK_fraction"] = scr_grp["T_NK_cells"] / scr_grp["total_cells"]

# 3) define top quartile in each modality (using full cohorts)
q_vis = vis_grp["Visium_TILP_fraction"].quantile(0.75)
q_scr = scr_grp["T_NK_fraction"].quantile(0.75)

vis_grp["Visium_hi"] = vis_grp["Visium_TILP_fraction"] >= q_vis
scr_grp["snRNA_hi"]  = scr_grp["T_NK_fraction"] >= q_scr

print(f"Visium TIL-high (>= Q3={q_vis:.4f} fraction): {int(vis_grp['Visium_hi'].sum())}/{len(vis_grp)}")
print(f"snRNA  T_NK-high (>= Q3={q_scr:.4f} fraction): {int(scr_grp['snRNA_hi'].sum())}/{len(scr_grp)}")

# 4) intersect Visium and snRNA on sample ID
merged = vis_grp.merge(
    scr_grp,
    left_on="vis_id",
    right_on="scr_id",
    how="inner",
)

if merged.empty:
    raise RuntimeError("No intersection between Visium and snRNA IDs.")

merged["both_hi"] = merged["Visium_hi"] & merged["snRNA_hi"]

n_both_hi   = int(merged["both_hi"].sum())
n_total_int = len(merged)
print(f"Intersection with both modalities: {n_total_int} samples")
print(f"High in both modalities: {n_both_hi}/{n_total_int}")

# 5) add clinical data and define groups
clin_subset = clin_out[[CLIN_ID_COL, TIME_COL, EVENT_COL]].copy()
merged = merged.merge(
    clin_subset,
    left_on="vis_id",
    right_on=CLIN_ID_COL,
    how="inner",
)

merged = merged.dropna(subset=[TIME_COL, EVENT_COL])

merged["TIL_group"] = np.where(merged["both_hi"], "TIL_both_high", "Other")

group_counts = merged["TIL_group"].value_counts()
print("\nGroup counts (with survival data):")
print(group_counts)

if group_counts.get("TIL_both_high", 0) < 5 or group_counts.get("Other", 0) < 5:
    print("Warning: very small group sizes, KM curves may be unstable.")

# 6) KM + logrank
high_mask = merged["TIL_group"] == "TIL_both_high"
low_mask  = merged["TIL_group"] == "Other"

T_high = merged.loc[high_mask, TIME_COL].astype(float)
E_high = merged.loc[high_mask, EVENT_COL].astype(int)

T_low  = merged.loc[low_mask, TIME_COL].astype(float)
E_low  = merged.loc[low_mask, EVENT_COL].astype(int)

kmf_high = KaplanMeierFitter()
kmf_low  = KaplanMeierFitter()

plt.figure(figsize=(7, 6))

kmf_high.fit(T_high, event_observed=E_high, label="TIL-high in both (Visium ∧ snRNA)")
kmf_high.plot(ci_show=True, ci_alpha=0.25)

kmf_low.fit(T_low, event_observed=E_low, label="Other patients")
kmf_low.plot(ci_show=True, ci_alpha=0.25)

plt.xlabel(TIME_COL)
plt.ylabel("Survival probability")
plt.title(PLOT_TITLE)
plt.grid(False)
plt.tight_layout()
plt.show()

lr = logrank_test(T_high, T_low, event_observed_A=E_high, event_observed_B=E_low)
print(f"\nLogrank test p value: {lr.p_value:.3e}")

# Survival analysis based on immune invasion module

### Define 3 module thresholds

In [ ]:
# %% module definitions (celltype_level4_curated space)

MODULES_L4 = {
    "broad": [
        "Microglia_intermediate_reactive",
        "NK_nos",
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
        "Mast_cell",
        "Plasma_cell",
        "Plasmablast",
        "Perivascular_fibroblast_cycling",
        "Perivascular_fibroblast_type-I",
        "Pericyte_fibro-like",
    ],
    "intermediate": [
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
        "Mast_cell",
        "Plasma_cell",
        "Plasmablast",
        "Perivascular_fibroblast_cycling",
    ],
    "tight": [
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
    ],
}

for name, vals in MODULES_L4.items():
    print(f"{name}: {len(vals)} level4_curated types")

### Survival based on Visium spot positivity

In [ ]:
# %% Visium survival based on C2L module burden (full cell)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ---------- user parameters ----------
MODULE_VERSION = "tight"      # "broad", "intermediate", or "tight"
C2L_THRESHOLD  = 2          # cells per spot to call a spot module-positive

SPLIT_MODE     = "top_quartile"   # "top_quartile", "top_10pct", or "threshold"
CUSTOM_CUT     = 0.4             # used only if SPLIT_MODE == "threshold" (on fraction)

VISIUM_SAMPLE_COL = "sample"
CLIN_ID_COL       = "Tumour block ID for MOSAIC"

TIME_COL          = "os_years"    # or "pfs_years"
EVENT_COL         = "os_event"    # or "pfs_event"

PLOT_TITLE        = f"Visium survival, module={MODULE_VERSION}, C2L≥{C2L_THRESHOLD}"
# ------------------------------------

# 1) choose module (in celltype_level4_curated space)
module_curated = MODULES_L4[MODULE_VERSION]
print(f"Module '{MODULE_VERSION}' with {len(module_curated)} level4_curated labels.")

# 2) map curated level4 names to scanvi names and then to C2L columns
key = adata.uns["celltype_annotation_key"].copy()

# treat the index as scanvi level4 names; name it for clarity
if key.index.name is None:
    key.index.name = "celltype_level4_scanvi"

if "celltype_level4_curated" not in key.columns:
    raise KeyError(
        "Expected 'celltype_level4_curated' in adata.uns['celltype_annotation_key'].\n"
        f"Available columns: {key.columns.tolist()}"
    )

# which scanvi types belong to this module?
mask = key["celltype_level4_curated"].isin(module_curated)
base_types = key.index[mask].astype(str)

if len(base_types) == 0:
    raise RuntimeError(
        f"No scanvi level4 types map to module '{MODULE_VERSION}'. "
        f"Module labels: {module_curated}"
    )

# build C2L columns from these scanvi base types
c2l_cols = [
    f"meanscell_abundance_w_sf_{ct}"
    for ct in base_types
    if f"meanscell_abundance_w_sf_{ct}" in adata.obs.columns
]

if len(c2l_cols) == 0:
    raise RuntimeError(
        f"No C2L columns found in adata.obs for module '{MODULE_VERSION}'. "
        f"Scanvi base types: {list(base_types)}"
    )

print(f"C2L columns used ({len(c2l_cols)}): {c2l_cols[:8]}{' ...' if len(c2l_cols) > 8 else ''}")

# 3) per-spot module count and module-positive flag
module_counts = adata.obs[c2l_cols].sum(axis=1).astype(float)
module_positive = module_counts >= C2L_THRESHOLD

# 4) per-patient module+ fraction
sample_ids = adata.obs[VISIUM_SAMPLE_COL].astype(str)
patient_ids = sample_ids.str.replace("_vis$", "", regex=True)

df_vis = pd.DataFrame(
    {
        "patient_id": patient_ids.values,
        "module_positive": module_positive.values,
    }
)
df_vis["Total_Spots"] = 1

vis_burden = (
    df_vis.groupby("patient_id", as_index=False)
          .agg(
              module_pos=("module_positive", "sum"),
              total_spots=("Total_Spots", "sum"),
          )
)
vis_burden["module_fraction"] = vis_burden["module_pos"] / vis_burden["total_spots"]

print(f"Patients with Visium module data: {vis_burden.shape[0]}")

# 5) join clinical data
clin_subset = clin_out[[CLIN_ID_COL, TIME_COL, EVENT_COL]].copy()
vis_clin = vis_burden.merge(
    clin_subset,
    left_on="patient_id",
    right_on=CLIN_ID_COL,
    how="inner",
)
vis_clin = vis_clin.dropna(subset=[TIME_COL, EVENT_COL, "module_fraction"])

print(f"Patients with Visium module + survival data: {vis_clin.shape[0]}")

# 6) define high vs low based on module_fraction
if SPLIT_MODE == "top_quartile":
    cutoff = vis_clin["module_fraction"].quantile(0.75)
elif SPLIT_MODE == "top_10pct":
    cutoff = vis_clin["module_fraction"].quantile(0.90)
elif SPLIT_MODE == "threshold":
    cutoff = float(CUSTOM_CUT)
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile', 'top_10pct', or 'threshold'.")

vis_clin["Module_group"] = np.where(
    vis_clin["module_fraction"] >= cutoff,
    "Module_high",
    "Module_low",
)

group_counts = vis_clin["Module_group"].value_counts()
print(f"\nCutoff on module_fraction: {cutoff:.4g}")
print("Group counts (Visium):")
print(group_counts)

# 7) Kaplan–Meier + logrank
high_mask = vis_clin["Module_group"] == "Module_high"
low_mask  = vis_clin["Module_group"] == "Module_low"

T_high = vis_clin.loc[high_mask, TIME_COL].astype(float)
E_high = vis_clin.loc[high_mask, EVENT_COL].astype(int)

T_low  = vis_clin.loc[low_mask, TIME_COL].astype(float)
E_low  = vis_clin.loc[low_mask, EVENT_COL].astype(int)

kmf_high = KaplanMeierFitter()
kmf_low  = KaplanMeierFitter()

plt.figure(figsize=(7, 6))

kmf_high.fit(T_high, event_observed=E_high, alpha=0.1, label=f"{MODULE_VERSION} module high (Visium)")
kmf_high.plot(ci_show=True, ci_alpha=0.25)

kmf_low.fit(T_low, event_observed=E_low, alpha=0.1, label=f"{MODULE_VERSION} module low (Visium)")
kmf_low.plot(ci_show=True, ci_alpha=0.25)

plt.xlabel(TIME_COL)
plt.ylabel("Survival probability")
plt.title(PLOT_TITLE)
plt.grid(False)
plt.tight_layout()
plt.show()

lr = logrank_test(T_high, T_low, event_observed_A=E_high, event_observed_B=E_low)
print(f"\nVisium logrank p value: {lr.p_value:.3e}")

### Survival based on single cell abundances

In [ ]:
# %% snRNA survival based on module cell fraction (level4_curated, 90% CI)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ---------- user parameters ----------
MODULE_VERSION = "broad"          # "broad", "intermediate", or "tight"

LEVEL_COL      = "celltype_level4_curated"
SCRNA_ID_COL   = "orig.ident"

CLIN_ID_COL    = "Tumour block ID for MOSAIC"
TIME_COL       = "os_years"       # or "pfs_years"
EVENT_COL      = "os_event"       # or "pfs_event"

SPLIT_MODE     = "threshold"   # "top_quartile", "top_10pct", or "threshold"
CUSTOM_CUT     = 0.1             # used only if SPLIT_MODE == "threshold" (on fraction)

PLOT_TITLE     = f"snRNA survival, module={MODULE_VERSION}"
# ------------------------------------

# 1) basic sanity checks
for col in [LEVEL_COL, SCRNA_ID_COL]:
    if col not in adata_sc.obs.columns:
        raise KeyError(f"'{col}' not in adata_sc.obs.columns")

for col in [CLIN_ID_COL, TIME_COL, EVENT_COL]:
    if col not in clin_out.columns:
        raise KeyError(f"'{col}' not in clin_out.columns")

if MODULE_VERSION not in MODULES_L4:
    raise KeyError(f"MODULE_VERSION '{MODULE_VERSION}' not in MODULES_L4 keys {list(MODULES_L4.keys())}")

module_curated = MODULES_L4[MODULE_VERSION]
print(f"Module '{MODULE_VERSION}' with {len(module_curated)} level4_curated labels.")

# 2) per-cell membership
obs_sc = adata_sc.obs[[SCRNA_ID_COL, LEVEL_COL]].copy()
obs_sc = obs_sc.dropna(subset=[LEVEL_COL])

levels_present = pd.Index(obs_sc[LEVEL_COL].unique())
mod_in_data = levels_present.intersection(module_curated)
mod_missing = pd.Index(module_curated).difference(levels_present)

print(f"Module labels present in data: {len(mod_in_data)}")
if len(mod_missing):
    print(f"Module labels missing from data (ignored): {list(mod_missing)[:10]}{' ...' if len(mod_missing) > 10 else ''}")

obs_sc["is_module"] = obs_sc[LEVEL_COL].isin(mod_in_data)

# 3) per-sample module fraction
scrna_burden = (
    obs_sc.groupby(SCRNA_ID_COL, as_index=False)
          .agg(
              module_cells=("is_module", "sum"),
              total_cells=(LEVEL_COL, "size"),
          )
)
scrna_burden["module_fraction_sc"] = scrna_burden["module_cells"] / scrna_burden["total_cells"]

print(f"Samples with snRNA module data: {scrna_burden.shape[0]}")

# 4) join clinical (orig.ident assumed to match Tumour block ID)
scrna_burden = scrna_burden.rename(columns={SCRNA_ID_COL: CLIN_ID_COL})

clin_subset = clin_out[[CLIN_ID_COL, TIME_COL, EVENT_COL]].copy()
sc_clin = scrna_burden.merge(clin_subset, on=CLIN_ID_COL, how="inner")
sc_clin = sc_clin.dropna(subset=[TIME_COL, EVENT_COL, "module_fraction_sc"])

print(f"Samples with snRNA module + survival data: {sc_clin.shape[0]}")

# 5) define high vs low based on module_fraction_sc
if SPLIT_MODE == "top_quartile":
    cutoff_sc = sc_clin["module_fraction_sc"].quantile(0.75)
elif SPLIT_MODE == "top_10pct":
    cutoff_sc = sc_clin["module_fraction_sc"].quantile(0.90)
elif SPLIT_MODE == "threshold":
    cutoff_sc = float(CUSTOM_CUT)
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile', 'top_10pct', or 'threshold'.")

sc_clin["Module_group"] = np.where(
    sc_clin["module_fraction_sc"] >= cutoff_sc,
    "Module_high",
    "Module_low",
)

group_counts_sc = sc_clin["Module_group"].value_counts()
print(f"\nCutoff on module_fraction_sc: {cutoff_sc:.4g}")
print("Group counts (snRNA):")
print(group_counts_sc)

# 6) KM + logrank with 90% confidence intervals
high_mask = sc_clin["Module_group"] == "Module_high"
low_mask  = sc_clin["Module_group"] == "Module_low"

T_high = sc_clin.loc[high_mask, TIME_COL].astype(float)
E_high = sc_clin.loc[high_mask, EVENT_COL].astype(int)

T_low  = sc_clin.loc[low_mask, TIME_COL].astype(float)
E_low  = sc_clin.loc[low_mask, EVENT_COL].astype(int)

# alpha here is confidence level (0.90 for 90% CI)
kmf_high = KaplanMeierFitter(alpha=0.10)
kmf_low  = KaplanMeierFitter(alpha=0.10)

plt.figure(figsize=(7, 6))

kmf_high.fit(T_high, event_observed=E_high, label=f"{MODULE_VERSION} module high (snRNA)")
kmf_high.plot(ci_show=True) 

kmf_low.fit(T_low, event_observed=E_low, label=f"{MODULE_VERSION} module low (snRNA)")
kmf_low.plot(ci_show=True)

plt.xlabel(TIME_COL)
plt.ylabel("Survival probability")
plt.title(PLOT_TITLE)
plt.grid(False)
plt.tight_layout()
plt.show()

lr = logrank_test(T_high, T_low, event_observed_A=E_high, event_observed_B=E_low)
print(f"\nsnRNA logrank p value: {lr.p_value:.3e}")

# Cox and RMST survival modeling

### Visium survival analysis

In [ ]:
# %% Visium T_NK survival: Cox + RMST with Q75/Q90 splits

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import restricted_mean_survival_time

# ---------------- user variables ----------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]        # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2.0             # C2L per spot cutoff for TIL_positive

TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"

RMST_TAU_YEARS = 3.0             # horizon for RMST
SPLIT_QUANTILES = [0.75, 0.90]   # test top quartile and top decile
# ------------------------------------------------

# basic checks
if "celltype_annotation_key" not in adata.uns:
    raise KeyError("adata.uns['celltype_annotation_key'] not found.")
for col in [TIME_COL_OS, EVENT_COL_OS, TIME_COL_PFS, EVENT_COL_PFS, PATIENT_ID_COL_CLIN]:
    if col not in clin_out.columns:
        raise KeyError(f"'{col}' not found in clin_out.columns.")

# 1. identify T_NK-related C2L columns at level2_curated
key = adata.uns["celltype_annotation_key"].copy()
key.index = key.index.astype(str)
if LEVEL_COL not in key.columns:
    raise KeyError(f"'{LEVEL_COL}' not found in celltype_annotation_key.")

lvl4_index = key.index
lvl2_vals = key[LEVEL_COL].astype(str)

tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]
if tnk_level4.empty:
    raise ValueError(f"No level4 entries mapped to level2 {TIL_GROUPS} in '{LEVEL_COL}'.")

c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_level4]
missing = [c for c in c2l_cols if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing expected C2L columns in adata.obs: {missing[:5]} ...")

# 2. per-spot TIL count and positivity (exactly as previous KM script)
obs = adata.obs.copy()
obs["TIL_count"] = obs[c2l_cols].sum(axis=1)
obs["TIL_positive"] = obs["TIL_count"] >= TIL_THRESHOLD

# 3. per-sample TIL stats
if "sample" not in obs.columns:
    raise KeyError("adata.obs['sample'] not found.")

til_stats_sample = obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]

# 4. map samples to patient IDs and aggregate to tumour-block level
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]

# 5. merge with clinical
clin_map = clin_out[
    [PATIENT_ID_COL_CLIN, TIME_COL_OS, EVENT_COL_OS, TIME_COL_PFS, EVENT_COL_PFS]
].copy()
clin_map = clin_map.rename(columns={PATIENT_ID_COL_CLIN: "patient_id"})

vis_surv = pd.merge(til_stats_patient.reset_index(), clin_map, on="patient_id", how="inner")

# drop rows missing survival
vis_surv_os = vis_surv.dropna(subset=[TIME_COL_OS, EVENT_COL_OS]).copy()
vis_surv_pfs = vis_surv.dropna(subset=[TIME_COL_PFS, EVENT_COL_PFS]).copy()

print(f"Patients with OS data:  {vis_surv_os.shape[0]}")
print(f"Patients with PFS data: {vis_surv_pfs.shape[0]}")

# 6. helper: RMST at tau
def _rmst(time, event, tau):
    km = KaplanMeierFitter()
    km.fit(time, event_observed=event)
    return float(restricted_mean_survival_time(km, t=tau))

results = []

for endpoint, df, time_col, event_col in [
    ("OS", vis_surv_os, TIME_COL_OS, EVENT_COL_OS),
    ("PFS", vis_surv_pfs, TIME_COL_PFS, EVENT_COL_PFS),
]:
    if df.empty:
        continue

    # continuous Cox with standardized TIL_fraction
    burden = df["TIL_fraction"].astype(float)
    df_cox = df[[time_col, event_col]].copy()
    sd = burden.std(ddof=0)
    if sd == 0:
        hr, hr_low, hr_high, p_cox = np.nan, np.nan, np.nan, np.nan
    else:
        df_cox["TIL_fraction_std"] = (burden - burden.mean()) / sd
        cph = CoxPHFitter()
        cph.fit(df_cox, duration_col=time_col, event_col=event_col)
        summ = cph.summary.loc["TIL_fraction_std"]
        hr = float(np.exp(summ["coef"]))
        hr_low = float(np.exp(summ["coef lower 95%"]))
        hr_high = float(np.exp(summ["coef upper 95%"]))
        p_cox = float(summ["p"])

    # store continuous model row (no split)
    results.append({
        "modality": "Visium",
        "feature": "T_NK",
        "n": int(df.shape[0]),
        "burden_mean": float(burden.mean()),
        "burden_std": float(sd),
        "split_type": "continuous",
        "split_param": None,
        "cutoff": None,
        "n_high": None,
        "n_low": None,
        "cox_HR_per_SD": hr,
        "cox_HR_CI95_low": hr_low,
        "cox_HR_CI95_high": hr_high,
        "cox_p": p_cox,
        "rmst_high": None,
        "rmst_low": None,
        "rmst_diff": None,
        "logrank_p": None,
        "endpoint": endpoint,
        "tau_years": RMST_TAU_YEARS,
    })

    # quantile-based high/low splits + RMST + log-rank
    for q in SPLIT_QUANTILES:
        cutoff = burden.quantile(q)
        groups = burden >= cutoff
        df_high = df.loc[groups]
        df_low = df.loc[~groups]

        if df_high.empty or df_low.empty:
            rmst_high = rmst_low = rmst_diff = logrank_p = np.nan
        else:
            # RMST
            rmst_high = _rmst(df_high[time_col], df_high[event_col], RMST_TAU_YEARS)
            rmst_low = _rmst(df_low[time_col], df_low[event_col], RMST_TAU_YEARS)
            rmst_diff = rmst_high - rmst_low

            # log-rank
            lr = logrank_test(
                df_high[time_col],
                df_low[time_col],
                event_observed_A=df_high[event_col],
                event_observed_B=df_low[event_col],
            )
            logrank_p = float(lr.p_value)

        results.append({
            "modality": "Visium",
            "feature": "T_NK",
            "n": int(df.shape[0]),
            "burden_mean": float(burden.mean()),
            "burden_std": float(sd),
            "split_type": f"q{int(q*100)}",
            "split_param": q,
            "cutoff": float(cutoff),
            "n_high": int(groups.sum()),
            "n_low": int((~groups).sum()),
            "cox_HR_per_SD": hr,
            "cox_HR_CI95_low": hr_low,
            "cox_HR_CI95_high": hr_high,
            "cox_p": p_cox,
            "rmst_high": rmst_high,
            "rmst_low": rmst_low,
            "rmst_diff": rmst_diff,
            "logrank_p": logrank_p,
            "endpoint": endpoint,
            "tau_years": RMST_TAU_YEARS,
        })

vis_tnk_surv_summary = pd.DataFrame(results)
print("\nVisium T_NK survival summary:")
display(vis_tnk_surv_summary)

vis_tnk_surv_summary.to_csv("Visium_TNK_survival_Cox_RMST_summary.csv", index=False)

In [ ]:
# %% Visium invasive modules (Broad / Intermediate / Tight):
#     Cox + RMST survival summary using C2L module fractions

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import restricted_mean_survival_time
from IPython.display import display

# ---------------- user variables ----------------
MODULE_DEFS = {
    "Module_broad": [
        "Microglia_intermediate_reactive",
        "NK_nos",
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
        "Mast_cell",
        "Plasma_cell",
        "Plasmablast",
        "Perivascular_fibroblast_cycling",
        "Perivascular_fibroblast_type-I",
        "Pericyte_fibro-like",
    ],
    "Module_intermediate": [
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
        "Mast_cell",
        "Plasma_cell",
        "Plasmablast",
        "Perivascular_fibroblast_cycling",
    ],
    "Module_tight": [
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
    ],
}

LEVEL4_CUR_COL  = "celltype_level4_curated"     # in adata.uns["celltype_annotation_key"]
C2L_PREFIX      = "meanscell_abundance_w_sf_"   # C2L columns in adata.obs
SPOT_THRESHOLD  = 2.0                           # min C2L "cells" per spot to call module-positive

PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"
TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

RMST_TAU_YEARS  = 3.0
SPLIT_QUANTILES = [0.75, 0.90]                  # 75th and 90th percentiles
# ------------------------------------------------

def normalize_id(series: pd.Series) -> pd.Series:
    """Make sample / patient IDs comparable (strip _vis and whitespace)."""
    return (
        series.astype(str)
        .str.replace("_vis", "", regex=False)
        .str.strip()
    )

def _rmst(time, event, tau):
    km = KaplanMeierFitter()
    km.fit(time, event_observed=event)
    return float(restricted_mean_survival_time(km, t=tau))

# sanity checks
if "celltype_annotation_key" not in adata.uns:
    raise KeyError("adata.uns['celltype_annotation_key'] not found.")

for col in [PATIENT_ID_COL_CLIN, TIME_COL_OS, EVENT_COL_OS, TIME_COL_PFS, EVENT_COL_PFS]:
    if col not in clin_out.columns:
        raise KeyError(f"'{col}' not found in clin_out.columns.")

if "sample" not in adata.obs.columns:
    raise KeyError("adata.obs['sample'] not found in adata.obs.columns.")

# --- annotation key and C2L mapping ---
key = adata.uns["celltype_annotation_key"].copy()
key.index = key.index.astype(str)

if LEVEL4_CUR_COL not in key.columns:
    raise KeyError(f"'{LEVEL4_CUR_COL}' not found in celltype_annotation_key columns.")

key["c2l_col"] = C2L_PREFIX + key.index

# --- spot-level base with normalized patient_id ---
obs_base = adata.obs[["sample"]].copy()
obs_base["patient_id"] = normalize_id(obs_base["sample"])

# total spots per patient
total_spots_pat = (
    obs_base.groupby("patient_id", observed=True)
    .size()
    .rename("total_spots")
)

# --- clinical mapping with normalized patient_id ---
clin_map = clin_out[
    [PATIENT_ID_COL_CLIN, TIME_COL_OS, EVENT_COL_OS, TIME_COL_PFS, EVENT_COL_PFS]
].copy()
clin_map["patient_id"] = normalize_id(clin_map[PATIENT_ID_COL_CLIN])
clin_map = clin_map.drop(columns=[PATIENT_ID_COL_CLIN])

print("Total unique patients in Visium (any spots):", len(total_spots_pat))
print("Total unique patients in clinical table:", clin_map["patient_id"].nunique())

results = []

for module_name, member_labels in MODULE_DEFS.items():
    print(f"\n{module_name}:")
    print(f"  defined members: {len(member_labels)}")

    # which curated level4 labels belong to this module
    key_mod = key[key[LEVEL4_CUR_COL].isin(member_labels)].copy()
    matched_labels = sorted(key_mod[LEVEL4_CUR_COL].unique())
    missing_labels = sorted(set(member_labels) - set(matched_labels))

    print(f"  matched in key:  {len(matched_labels)}")
    if missing_labels:
        print(f"  missing labels (first few): {missing_labels[:5]}")

    if key_mod.empty:
        print("  [SKIP] no level4_curated labels from this module found in key.")
        continue

    # corresponding C2L columns present in adata.obs
    c2l_cols = [c for c in key_mod["c2l_col"].tolist() if c in adata.obs.columns]
    print(f"  matching C2L columns in adata.obs: {len(c2l_cols)}")

    if not c2l_cols:
        print("  [SKIP] no C2L columns found for module in adata.obs.")
        continue

    # per-spot module abundance and positivity
    mod_count = adata.obs[c2l_cols].sum(axis=1)
    mod_pos = mod_count >= SPOT_THRESHOLD

    df_mod = pd.DataFrame({
        "patient_id": obs_base["patient_id"].values,
        "module_spot_pos": mod_pos.astype(int).values,
    })

    # module spots per patient
    mod_stats = (
        df_mod
        .groupby("patient_id", observed=True)
        .agg(module_spots=("module_spot_pos", "sum"))
    )

    # attach total spots and module fraction
    mod_stats["total_spots"] = total_spots_pat.reindex(mod_stats.index)
    mod_stats["module_fraction"] = mod_stats["module_spots"] / mod_stats["total_spots"]

    # drop any patients with missing or zero total_spots
    mod_stats = mod_stats.dropna(subset=["total_spots"])
    mod_stats = mod_stats.loc[mod_stats["total_spots"] > 0]

    print("  patients with module stats:", len(mod_stats))

    # merge with clinical data
    base = mod_stats.reset_index().merge(clin_map, on="patient_id", how="inner")
    print("  patients with module + survival:", base["patient_id"].nunique())

    for endpoint, time_col, event_col in [
        ("OS", TIME_COL_OS, EVENT_COL_OS),
        ("PFS", TIME_COL_PFS, EVENT_COL_PFS),
    ]:
        df = base.dropna(subset=[time_col, event_col]).copy()
        if df.empty:
            print(f"  [SKIP] {endpoint}: no patients with survival data after dropna.")
            continue

        burden = df["module_fraction"].astype(float)
        n_pat = int(df.shape[0])
        mean_b = float(burden.mean())
        sd_b = float(burden.std(ddof=0))

        # continuous Cox per SD of module_fraction
        if sd_b == 0:
            hr = hr_low = hr_high = p_cox = np.nan
        else:
            df_cox = df[[time_col, event_col]].copy()
            df_cox["module_std"] = (burden - mean_b) / sd_b
            cph = CoxPHFitter()
            cph.fit(df_cox, duration_col=time_col, event_col=event_col)
            summ = cph.summary.loc["module_std"]
            hr = float(np.exp(summ["coef"]))
            hr_low = float(np.exp(summ["coef lower 95%"]))
            hr_high = float(np.exp(summ["coef upper 95%"]))
            p_cox = float(summ["p"])

        # continuous row (no split)
        results.append({
            "modality": "Visium",
            "feature": module_name,
            "n": n_pat,
            "burden_mean": mean_b,
            "burden_std": sd_b,
            "split_type": "continuous",
            "split_param": None,
            "cutoff": None,
            "n_high": None,
            "n_low": None,
            "cox_HR_per_SD": hr,
            "cox_HR_CI95_low": hr_low,
            "cox_HR_CI95_high": hr_high,
            "cox_p": p_cox,
            "rmst_high": None,
            "rmst_low": None,
            "rmst_diff": None,
            "logrank_p": None,
            "endpoint": endpoint,
            "tau_years": RMST_TAU_YEARS,
        })

        # Q75 and Q90 splits on module_fraction
        for q in SPLIT_QUANTILES:
            cutoff = float(burden.quantile(q))
            groups = burden >= cutoff
            df_high = df.loc[groups]
            df_low = df.loc[~groups]

            n_high = int(df_high.shape[0])
            n_low = int(df_low.shape[0])

            if df_high.empty or df_low.empty:
                rmst_high = rmst_low = rmst_diff = logrank_p = np.nan
            else:
                rmst_high = _rmst(df_high[time_col], df_high[event_col], RMST_TAU_YEARS)
                rmst_low = _rmst(df_low[time_col], df_low[event_col], RMST_TAU_YEARS)
                rmst_diff = rmst_high - rmst_low

                lr = logrank_test(
                    df_high[time_col],
                    df_low[time_col],
                    event_observed_A=df_high[event_col],
                    event_observed_B=df_low[event_col],
                )
                logrank_p = float(lr.p_value)

            results.append({
                "modality": "Visium",
                "feature": module_name,
                "n": n_pat,
                "burden_mean": mean_b,
                "burden_std": sd_b,
                "split_type": f"q{int(q*100)}",
                "split_param": q,
                "cutoff": cutoff,
                "n_high": n_high,
                "n_low": n_low,
                "cox_HR_per_SD": hr,
                "cox_HR_CI95_low": hr_low,
                "cox_HR_CI95_high": hr_high,
                "cox_p": p_cox,
                "rmst_high": rmst_high,
                "rmst_low": rmst_low,
                "rmst_diff": rmst_diff,
                "logrank_p": logrank_p,
                "endpoint": endpoint,
                "tau_years": RMST_TAU_YEARS,
            })

vis_modules_surv_summary = pd.DataFrame(results)
print("\nVisium invasive module survival summary (Broad / Intermediate / Tight):")
print(f"Rows in summary: {vis_modules_surv_summary.shape[0]}")
display(vis_modules_surv_summary)

vis_modules_surv_summary.to_csv(
    "Visium_invasive_modules_survival_Cox_RMST_summary.csv",
    index=False,
)

In [ ]:
# %% Survival RMST for T_NK and invasive modules (Visium, C2L-based)

import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import restricted_mean_survival_time

# ---------------- user parameters ----------------
LEVEL2_COL = "celltype_level2_curated"
LEVEL4_CURATED_COL = "celltype_level4_curated"

# TIL definition (Visium, via C2L)
TIL_LEVEL2 = ["T_NK"]     # level2_curated labels treated as T/NK
C2L_THRESHOLD = 2.0       # per-spot C2L cutoff for "positive"

# Clinical mapping
PATIENT_ID_CLIN = "Tumour block ID for MOSAIC"
TIME_COL_OS    = "os_years"
EVENT_COL_OS   = "os_event"
TIME_COL_PFS   = "pfs_years"
EVENT_COL_PFS  = "pfs_event"

# RMST horizons
TAU_LIST = [2.0, 3.0]     # years

# Module definitions in celltype_level4_curated space
MODULES_CURATED = {
    "Module_broad": [
        "Microglia_intermediate_reactive",
        "NK_nos",
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
        "Mast_cell",
        "Plasma_cell",
        "Plasmablast",
        "Perivascular_fibroblast_cycling",
        "Perivascular_fibroblast_type-I",
        "Pericyte_fibro-like",
    ],
    "Module_intermediate": [
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
        "Mast_cell",
        "Plasma_cell",
        "Plasmablast",
        "Perivascular_fibroblast_cycling",
    ],
    "Module_tight": [
        "cDC2",
        "pDC",
        "NK_KLRC1_pos",
        "Macrophage_C1Q_pos",
        "T_cell_IFN_response",
        "T_cell_CD4",
        "T_cell_CD8",
        "T_NK_cycling",
        "cDC1",
        "T_reg",
    ],
}
# -------------------------------------------------

# 1) Annotation key and helper from curated names to C2L columns
key = adata.uns["celltype_annotation_key"].copy()
key.index = key.index.astype(str)  # base celltype IDs (match C2L factors)

for col in [LEVEL2_COL, LEVEL4_CURATED_COL]:
    if col not in key.columns:
        raise KeyError(f"Column '{col}' not found in celltype_annotation_key columns: {key.columns.tolist()}")

def curated_to_c2l_cols(curated_list):
    """Given a list of level4_curated labels, return:
       - matched base factor names (index in key)
       - corresponding C2L obs column names
       - list of curated names that were not found
    """
    curated_arr = pd.Index(curated_list).astype(str)
    all_curated = key[LEVEL4_CURATED_COL].astype(str)

    mask = all_curated.isin(curated_arr)
    matched_idx = key.index[mask]

    matched_curated = set(all_curated[mask])
    all_curated_set = set(curated_arr)
    missing = sorted(all_curated_set - matched_curated)

    cols = [f"meanscell_abundance_w_sf_{ct}" for ct in matched_idx]
    return matched_idx.tolist(), cols, missing

# 2) T_NK C2L columns via level2_curated
mask_tnk = key[LEVEL2_COL].astype(str).isin(TIL_LEVEL2)
tnk_factors = key.index[mask_tnk]
tnk_c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_factors]

print(f"T_NK factors (level4_scanvi index) matched via {LEVEL2_COL}={TIL_LEVEL2}: {len(tnk_factors)}")

# 3) Module C2L column sets
module_c2l = {}
for mod_name, curated_list in MODULES_CURATED.items():
    matched, cols, missing = curated_to_c2l_cols(curated_list)
    print(f"\n{mod_name}:")
    print(f"  defined members: {len(curated_list)}")
    print(f"  matched in key:  {len(matched)}")
    print(f"  matching C2L columns in adata.obs: {len([c for c in cols if c in adata.obs.columns])}")
    if missing:
        print(f"  missing curated labels (not in key[{LEVEL4_CURATED_COL}]): {', '.join(missing)}")
    module_c2l[mod_name] = cols

# 4) Build a working obs table with patient IDs and C2L counts
obs = adata.obs.copy()

if "sample" not in obs.columns:
    raise KeyError("adata.obs['sample'] is required but not found.")

samples = obs["sample"].astype(str)
patient_ids = samples.str.replace("_vis$", "", regex=True)
obs = obs.assign(patient_id=patient_ids)

# sanity check C2L columns
for col in tnk_c2l_cols:
    if col not in obs.columns:
        raise KeyError(f"Missing T_NK C2L column in adata.obs: {col}")

for mod_name, cols in module_c2l.items():
    for col in cols:
        if col not in obs.columns:
            raise KeyError(f"Missing C2L column for {mod_name} in adata.obs: {col}")

# T_NK counts and positives
obs["TNK_count"] = obs[tnk_c2l_cols].sum(axis=1)
obs["TNK_positive"] = obs["TNK_count"] >= C2L_THRESHOLD

# Module counts and positives
for mod_name, cols in module_c2l.items():
    count_col = f"{mod_name}_count"
    pos_col = f"{mod_name}_positive"
    obs[count_col] = obs[cols].sum(axis=1)
    obs[pos_col] = obs[count_col] >= C2L_THRESHOLD

# 5) Aggregate to patient-level fractions (spot-level positives / total spots)
def positive_fraction(obs_df, pos_col):
    tmp = (
        obs_df.groupby("patient_id")[pos_col]
        .agg(spots_positive="sum", total_spots="count")
    )
    tmp["fraction"] = tmp["spots_positive"] / tmp["total_spots"]
    return tmp

tnk_stats = positive_fraction(obs, "TNK_positive").rename(
    columns={"spots_positive": "TNK_spots", "fraction": "TNK_fraction"}
)

mod_stats = {}
for mod_name in MODULES_CURATED.keys():
    pos_col = f"{mod_name}_positive"
    tmp = positive_fraction(obs, pos_col).rename(
        columns={
            "spots_positive": f"{mod_name}_spots",
            "fraction": f"{mod_name}_fraction",
        }
    )
    mod_stats[mod_name] = tmp

# Combine all fractions into one patient-level table
vis_burden = tnk_stats.copy()
for mod_name, tmp in mod_stats.items():
    vis_burden = vis_burden.join(tmp[[f"{mod_name}_fraction"]], how="outer")

vis_burden.index.name = "patient_id"

print("\nPatient-level burden table (head):")
print(vis_burden.head())

# 6) Merge with clinical data
needed_clin_cols = [PATIENT_ID_CLIN, TIME_COL_OS, EVENT_COL_OS, TIME_COL_PFS, EVENT_COL_PFS]
for c in needed_clin_cols:
    if c not in clin_out.columns:
        raise KeyError(f"Required clinical column '{c}' not found in clin_out.")

clin_map = clin_out[needed_clin_cols].copy()
clin_map = clin_map.dropna(subset=[PATIENT_ID_CLIN])
clin_map[PATIENT_ID_CLIN] = clin_map[PATIENT_ID_CLIN].astype(str)

vis_burden = vis_burden.reset_index()
vis_burden["patient_id_clin"] = vis_burden["patient_id"].str.replace("_vis$", "", regex=True)

vis_surv = vis_burden.merge(
    clin_map,
    left_on="patient_id_clin",
    right_on=PATIENT_ID_CLIN,
    how="inner",
)

print(f"\nMerged Visium burden + clinical: {vis_surv.shape[0]} patients with survival data.")

# 7) RMST helper: high vs low (q75 split) for a given feature, endpoint, and tau
def rmst_high_low(df, burden_col, time_col, event_col, tau, quantile=0.75):
    df = df.dropna(subset=[burden_col, time_col, event_col]).copy()
    if df.empty:
        return None

    cutoff = df[burden_col].quantile(quantile)
    df["group"] = np.where(df[burden_col] >= cutoff, "high", "low")

    n_high = int((df["group"] == "high").sum())
    n_low = int((df["group"] == "low").sum())

    # Require at least a handful of patients in each group
    if n_high < 5 or n_low < 5:
        return {
            "cutoff": float(cutoff),
            "n_high": n_high,
            "n_low": n_low,
            "rmst_high": np.nan,
            "rmst_low": np.nan,
            "rmst_diff": np.nan,
            "logrank_p": np.nan,
        }

    kmf_high = KaplanMeierFitter(alpha=0.1)
    kmf_low = KaplanMeierFitter(alpha=0.1)

    mask_high = df["group"] == "high"
    t_high = df.loc[mask_high, time_col]
    e_high = df.loc[mask_high, event_col]
    t_low = df.loc[~mask_high, time_col]
    e_low = df.loc[~mask_high, event_col]

    kmf_high.fit(t_high, event_observed=e_high)
    kmf_low.fit(t_low, event_observed=e_low)

    rmst_h = restricted_mean_survival_time(kmf_high, t=tau)
    rmst_l = restricted_mean_survival_time(kmf_low, t=tau)

    lr = logrank_test(
        t_high, t_low,
        event_observed_A=e_high,
        event_observed_B=e_low,
    )

    return {
        "cutoff": float(cutoff),
        "n_high": n_high,
        "n_low": n_low,
        "rmst_high": float(rmst_h),
        "rmst_low": float(rmst_l),
        "rmst_diff": float(rmst_h - rmst_l),
        "logrank_p": float(lr.p_value),
    }

# 8) Run RMST for T_NK + modules, OS + PFS, tau in TAU_LIST
features = {
    "T_NK": "TNK_fraction",
    "Module_broad": "Module_broad_fraction",
    "Module_intermediate": "Module_intermediate_fraction",
    "Module_tight": "Module_tight_fraction",
}

rows = []
for feat_name, col in features.items():
    if col not in vis_surv.columns:
        print(f"Warning: '{col}' not found in vis_surv, skipping {feat_name}.")
        continue

    for endpoint, time_col, event_col in [
        ("OS", TIME_COL_OS, EVENT_COL_OS),
        ("PFS", TIME_COL_PFS, EVENT_COL_PFS),
    ]:
        for tau in TAU_LIST:
            res = rmst_high_low(vis_surv, col, time_col, event_col, tau, quantile=0.75)
            if res is None:
                continue
            res.update({
                "feature": feat_name,
                "burden_col": col,
                "endpoint": endpoint,
                "tau_years": tau,
            })
            rows.append(res)

rmst_summary = pd.DataFrame(rows)[
    ["feature", "endpoint", "tau_years", "cutoff", "n_high", "n_low",
     "rmst_high", "rmst_low", "rmst_diff", "logrank_p", "burden_col"]
]

print("\nRMST summary (q75 split, tau in", TAU_LIST, "years):")
print(rmst_summary)

rmst_summary

In [ ]:
# %% Visium survival wrap-up: T_NK + immune modules (Cox + RMST)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.utils import restricted_mean_survival_time

# ---------------- user parameters ----------------
FEATURES = {
    "T_NK": "TNK_fraction",
    "Module_broad": "Module_broad_fraction",
    "Module_intermediate": "Module_intermediate_fraction",
    "Module_tight": "Module_tight_fraction",
}

ENDPOINTS = [
    ("OS", "os_years", "os_event"),
    ("PFS", "pfs_years", "pfs_event"),
]

Q_SPLIT = 0.75          # top quartile vs rest
TAU_LIST = [2.0, 3.0]   # RMST horizons (years)
CSV_PATH = "visium_TNK_and_modules_survival_summary.csv"
# -------------------------------------------------


def analyze_feature_visium(df, feature_name, burden_col, time_col, event_col,
                           q_split=0.75, tau_list=(2.0, 3.0)):
    """
    Univariate Cox (per SD) + RMST with q_split on burden_col, for one endpoint.
    Returns a list of dict rows (one per tau).
    """
    # subset and clean
    cols_needed = [burden_col, time_col, event_col]
    df_sub = df[cols_needed].copy().dropna()
    n = df_sub.shape[0]
    if n == 0:
        return []

    burden = df_sub[burden_col].astype(float)
    time = df_sub[time_col].astype(float)
    event = df_sub[event_col].astype(int)

    mean_b = float(burden.mean())
    std_b = float(burden.std(ddof=0))

    # Cox per SD (standardized predictor)
    if std_b > 0:
        z = (burden - mean_b) / std_b
        df_cox = pd.DataFrame(
            {"time": time.values, "event": event.values, "burden_z": z.values}
        )

        cph = CoxPHFitter()
        cph.fit(df_cox, duration_col="time", event_col="event")

        hr = float(np.exp(cph.params_["burden_z"]))
        ci_low, ci_high = np.exp(
            cph.confidence_intervals_.loc["burden_z"].values
        )
        p_cox = float(cph.summary.loc["burden_z", "p"])

        try:
            ph_res = proportional_hazard_test(
                cph, df_cox, time_transform="rank"
            )
            ph_p = float(ph_res.summary.loc["burden_z", "p"])
        except Exception:
            ph_p = np.nan
    else:
        hr = np.nan
        ci_low = np.nan
        ci_high = np.nan
        p_cox = np.nan
        ph_p = np.nan

    # q_split on raw burden (not z-score)
    cut = float(burden.quantile(q_split))
    is_high = burden >= cut
    n_high = int(is_high.sum())
    n_low = int(n - n_high)

    rows = []
    for tau in tau_list:
        km_high = KaplanMeierFitter()
        km_low = KaplanMeierFitter()

        km_high.fit(
            time[is_high],
            event_observed=event[is_high],
            label=f"{feature_name} high",
        )
        km_low.fit(
            time[~is_high],
            event_observed=event[~is_high],
            label=f"{feature_name} low",
        )

        rmst_hi = float(restricted_mean_survival_time(km_high, t=tau))
        rmst_lo = float(restricted_mean_survival_time(km_low, t=tau))
        rmst_diff = rmst_hi - rmst_lo

        lr = logrank_test(
            time[is_high],
            time[~is_high],
            event_observed_A=event[is_high],
            event_observed_B=event[~is_high],
        )
        logrank_p = float(lr.p_value)

        rows.append(
            {
                "modality": "Visium",
                "feature": feature_name,
                "n": int(n),
                "burden_col": burden_col,
                "burden_mean": mean_b,
                "burden_std": std_b,
                "split_type": "q75",
                "split_param": q_split,
                "cutoff": cut,
                "n_high": n_high,
                "n_low": n_low,
                "cox_HR_per_SD": hr,
                "cox_HR_CI95_low": float(ci_low),
                "cox_HR_CI95_high": float(ci_high),
                "cox_p": p_cox,
                "ph_p": ph_p,
                "rmst_high": rmst_hi,
                "rmst_low": rmst_lo,
                "rmst_diff": rmst_diff,
                "logrank_p": logrank_p,
                "endpoint": time_col.replace("_years", "").upper(),  # OS / PFS
                "tau_years": float(tau),
            }
        )

    return rows


# --- main summary over all features / endpoints ---
all_rows = []

for feat_name, bcol in FEATURES.items():
    for endpoint_name, t_col, e_col in ENDPOINTS:
        # reuse same function, endpoint label will be overwritten to match t_col
        rows = analyze_feature_visium(
            vis_surv,
            feature_name=feat_name,
            burden_col=bcol,
            time_col=t_col,
            event_col=e_col,
            q_split=Q_SPLIT,
            tau_list=TAU_LIST,
        )
        # correct endpoint label explicitly
        for r in rows:
            r["endpoint"] = endpoint_name
        all_rows.extend(rows)

visium_surv_summary = pd.DataFrame(all_rows)

# save and print
visium_surv_summary.to_csv(CSV_PATH, index=False)
print(f"Saved Visium survival summary to: {CSV_PATH}")
print("\nVisium survival summary (head):")
display(visium_surv_summary.head(20))

# --- pick the most significant RMST (smallest logrank p at tau = 3 years) ---
mask_best = (
    visium_surv_summary["tau_years"].eq(3.0)
    & visium_surv_summary["logrank_p"].notna()
)
if mask_best.any():
    best_row = (
        visium_surv_summary.loc[mask_best]
        .sort_values("logrank_p")
        .iloc[0]
    )

    feat = best_row["feature"]
    endpoint = best_row["endpoint"]
    burden_col = best_row["burden_col"]
    cutoff = best_row["cutoff"]
    logrank_best = best_row["logrank_p"]

    print("\nMost significant RMST (Visium, tau=3):")
    print(
        f"  feature={feat}, endpoint={endpoint}, "
        f"cutoff={cutoff:.4f}, logrank_p={logrank_best:.3e}"
    )

    # KM plot for the best feature / endpoint
    if endpoint == "OS":
        time_col = "os_years"
        event_col = "os_event"
        surv_label = "Overall survival"
    else:
        time_col = "pfs_years"
        event_col = "pfs_event"
        surv_label = "Progression-free survival"

    df_plot = vis_surv[[time_col, event_col, burden_col]].copy().dropna()
    time = df_plot[time_col].astype(float)
    event = df_plot[event_col].astype(int)
    burden = df_plot[burden_col].astype(float)

    is_high = burden >= cutoff

    km_high = KaplanMeierFitter()
    km_low = KaplanMeierFitter()

    km_high.fit(
        time[is_high],
        event_observed=event[is_high],
        label=f"{feat} high (≥ q75)",
        alpha=0.1,
    )
    km_low.fit(
        time[~is_high],
        event_observed=event[~is_high],
        label=f"{feat} low (< q75)",
        alpha=0.1,
    )

    plt.figure(figsize=(6, 5))
    km_high.plot(ci_show=True)
    km_low.plot(ci_show=True)
    plt.xlabel(f"{surv_label} (years)")
    plt.ylabel("Survival probability")
    plt.title(
        f"{surv_label} by {feat} burden\n"
        f"q75 split, logrank p = {logrank_best:.2e}"
    )
    plt.tight_layout()
    plt.show()
else:
    print("\nNo valid RMST rows found to plot (check visium_surv_summary).")

In [ ]:
# %% KM + shaded RMST area for T_NK (Visium, OS)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ---------------- user controls ----------------
LEVEL_COL      = "celltype_level2_curated"
TIL_GROUPS     = ["T_NK"]        # level2_curated group(s) treated as TIL
TIL_THRESHOLD  = 2.0             # C2L per spot cutoff for TIL_positive

BURDEN_METRIC  = "TIL_fraction"  # "TIL_spots" or "TIL_fraction"
SPLIT_MODE     = "top_quartile"  # "top_quartile" or "top_decile"

TIME_COL       = "os_years"
EVENT_COL      = "os_event"
PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"

# RMST horizons
TAUS           = [3.0]

# Aesthetics
FIGSIZE        = (6, 5)
LINEWIDTH      = 2.0
FONTSIZE       = 12
TICK_FONTSIZE  = 10
LEGEND_FONTSIZE = 10
ANNOT_FONTSIZE = 10
TEXT_X         = 0.55   # annotation position in axes coords
TEXT_Y         = 0.30
COLOR_HIGH     = "tab:red"
COLOR_LOW      = "tab:blue"
ALPHA_FILL     = 0.20
# ------------------------------------------------


def km_curve_truncated(kmf, tau):
    """Return (t, S(t)) for a KM curve, truncated or extended to tau."""
    surv = kmf.survival_function_
    t = surv.index.values.astype(float)
    y = surv.iloc[:, 0].values.astype(float)

    # Ensure we start at t=0, S(0)=1
    if t[0] > 0.0:
        t = np.insert(t, 0, 0.0)
        y = np.insert(y, 0, 1.0)

    if tau < t[-1]:
        mask = t <= tau
        t_tr = t[mask]
        y_tr = y[mask]
        if t_tr[-1] < tau:
            t_tr = np.append(t_tr, tau)
            y_tr = np.append(y_tr, y_tr[-1])
    else:
        t_tr = t.copy()
        y_tr = y.copy()
        if tau > t_tr[-1]:
            t_tr = np.append(t_tr, tau)
            y_tr = np.append(y_tr, y_tr[-1])

    return t_tr, y_tr


def rmst_from_curve(t, y):
    """Compute RMST = ∫_0^tau S(t) dt given step curve t, y (already truncated at tau)."""
    return float(np.trapz(y, t))


# 1. Identify level4 T_NK cell types via level2_curated
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in tnk_level4]

# 2. Spot-level TIL count and positivity
adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# 3. Sample-level TIL stats
til_stats_sample = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats_sample["TIL_fraction"] = (
    til_stats_sample["TIL_spots"] / til_stats_sample["total_spots"]
)

# 4. Map sample -> patient (strip "_vis") and aggregate to patient level
patient_ids = til_stats_sample.index.to_series().str.replace("_vis$", "", regex=True)
til_stats_sample = til_stats_sample.assign(patient_id=patient_ids.values)

til_stats_patient = til_stats_sample.groupby("patient_id").agg(
    TIL_spots=("TIL_spots", "sum"),
    total_spots=("total_spots", "sum"),
)
til_stats_patient["TIL_fraction"] = (
    til_stats_patient["TIL_spots"] / til_stats_patient["total_spots"]
)

# 5. Merge with clinical
til_clin = til_stats_patient.merge(
    clin_out,
    left_on="patient_id",
    right_on=PATIENT_ID_COL_CLIN,
    how="inner",
)

til_clin_valid = til_clin.dropna(subset=[TIME_COL, EVENT_COL])

# 6. Define TIL_high vs TIL_low based on burden
if BURDEN_METRIC not in ["TIL_spots", "TIL_fraction"]:
    raise ValueError("BURDEN_METRIC must be 'TIL_spots' or 'TIL_fraction'.")

burden = til_clin_valid[BURDEN_METRIC]

if SPLIT_MODE == "top_quartile":
    q = 0.75
    split_desc = "top quartile"
elif SPLIT_MODE == "top_decile":
    q = 0.90
    split_desc = "top decile"
else:
    raise ValueError("SPLIT_MODE must be 'top_quartile' or 'top_decile'.")

cutoff = burden.quantile(q)
til_clin_valid["TIL_group"] = np.where(burden >= cutoff, "TIL_high", "TIL_low")

print(f"Patients with survival data: {til_clin_valid.shape[0]}")
print(f"TIL definition: {LEVEL_COL} {TIL_GROUPS}, C2L threshold {TIL_THRESHOLD}")
print(f"Burden metric: {BURDEN_METRIC}")
print(f"Split: {split_desc} (cutoff {cutoff:.4f} on {BURDEN_METRIC})")
print(til_clin_valid["TIL_group"].value_counts(), "\n")

mask_high = til_clin_valid["TIL_group"] == "TIL_high"
mask_low  = til_clin_valid["TIL_group"] == "TIL_low"

t_high = til_clin_valid.loc[mask_high, TIME_COL].astype(float)
e_high = til_clin_valid.loc[mask_high, EVENT_COL].astype(int)

t_low  = til_clin_valid.loc[mask_low, TIME_COL].astype(float)
e_low  = til_clin_valid.loc[mask_low, EVENT_COL].astype(int)

max_time = float(til_clin_valid[TIME_COL].max())

# 7. Loop over tau values and make KM + shaded RMST plots
for tau in TAUS:
    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()

    kmf_high.fit(t_high, event_observed=e_high, label="TIL high (top quartile)")
    kmf_low.fit(t_low, event_observed=e_low, label="TIL low (remaining)")

    # Truncate curves at tau for plotting and RMST
    t_h, s_h = km_curve_truncated(kmf_high, tau)
    t_l, s_l = km_curve_truncated(kmf_low, tau)

    rmst_high = rmst_from_curve(t_h, s_h)
    rmst_low = rmst_from_curve(t_l, s_l)
    rmst_diff = rmst_high - rmst_low

    lr_res = logrank_test(t_high, t_low, event_observed_A=e_high, event_observed_B=e_low)
    p_val = lr_res.p_value

    print(f"tau = {tau:.1f} years:")
    print(f"  RMST high: {rmst_high:.3f} years")
    print(f"  RMST low:  {rmst_low:.3f} years")
    print(f"  ΔRMST (high - low): {rmst_diff:.3f} years")
    print(f"  log-rank p: {p_val:.3e}\n")

    fig, ax = plt.subplots(figsize=FIGSIZE)

    # Step curves without CI
    ax.step(t_h, s_h, where="post", color=COLOR_HIGH, linewidth=LINEWIDTH,
            label="TIL high (top quartile)")
    ax.step(t_l, s_l, where="post", color=COLOR_LOW, linewidth=LINEWIDTH,
            label="TIL low (remaining)")

    # Shade area under each curve up to tau
    ax.fill_between(t_h, s_h, 0.0, step="post", color=COLOR_HIGH, alpha=ALPHA_FILL)
    ax.fill_between(t_l, s_l, 0.0, step="post", color=COLOR_LOW, alpha=ALPHA_FILL)

    # Vertical line at tau
    ax.axvline(tau, linestyle="--", linewidth=1.0, color="gray", alpha=0.7)

    # Labels etc.
    ax.set_xlabel("Overall survival (years)", fontsize=FONTSIZE)
    ax.set_ylabel("Survival probability", fontsize=FONTSIZE)
    ax.set_title(f"OS by T_NK burden with RMST shading (tau = {tau:.1f} years)",
                 fontsize=FONTSIZE + 1)

    ax.tick_params(labelsize=TICK_FONTSIZE)

    leg = ax.legend(fontsize=LEGEND_FONTSIZE)
    for line in leg.get_lines():
        line.set_linewidth(LINEWIDTH)

    x_max = max(max_time, tau)
    ax.set_xlim(0.0, x_max)
    ax.set_ylim(0.0, 1.05)

    text_str = (
        f"RMST (τ = {tau:.1f} y)\n"
        f"TIL high: {rmst_high:.2f} y\n"
        f"TIL low:  {rmst_low:.2f} y\n"
        f"Δ = {rmst_diff:.2f} y\n"
        f"log-rank p = {p_val:.2e}"
    )
    ax.text(
        TEXT_X,
        TEXT_Y,
        text_str,
        transform=ax.transAxes,
        fontsize=ANNOT_FONTSIZE,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7, linewidth=0.0),
    )

    plt.tight_layout()
    plt.show()

# Correlation analysis between TILs and patient level features

### Build patient level variable table

In [ ]:
# %% Build PLV table (Option B: maximal table on MHC ∩ mutation ∩ clinical)
#     Now includes TIL4_tight_fraction_sc

import numpy as np
import pandas as pd

# ---------- parameters and paths ----------
LEVEL_COL = "celltype_level2_curated"
TIL_GROUPS = ["T_NK"]
TIL_THRESHOLD = 2.0
PATIENT_ID_COL_CLIN = "Tumour block ID for MOSAIC"

# Fixed cutoff from survival analysis (do NOT recompute)
TIL_Q75_CUTOFF = 0.04

BASE = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd"

PATH_L3 = f"{BASE}/results/scRNAseq/Cell_Proportions/sc_celltype_level3_proportions.csv"
PATH_L4 = f"{BASE}/results/scRNAseq/Cell_Proportions/sc_celltype_level4_proportions.csv"

PATH_GSEA_TUMOR   = f"{BASE}/results/scRNAseq/GSEA/pseudobulk_ulm_scores/ulm_scores_tumor.csv"
PATH_GSEA_MYELOID = f"{BASE}/results/scRNAseq/GSEA/pseudobulk_ulm_scores/ulm_scores_myeloid.csv"
PATH_GSEA_VASC    = f"{BASE}/results/scRNAseq/GSEA/pseudobulk_ulm_scores/ulm_scores_vascular.csv"
PATH_GSEA_TNK     = f"{BASE}/results/scRNAseq/GSEA/pseudobulk_ulm_scores/ulm_scores_tnk.csv"

PATH_MUT = f"{BASE}/results/WES/mutations/patient_gene_matrix.csv"
PATH_MHC = f"{BASE}/results/scratch/tmp_mhc_scores.parquet"

OUT_CSV = "patient_level_variables_TIL_and_PLVs.csv"
# ------------------------------------------


def add_root_id(series: pd.Series) -> pd.Series:
    """Strip trailing single lowercase letter from IDs (e.g. CH_G_164a -> CH_G_164)."""
    return series.astype(str).str.replace(r"[a-z]$", "", regex=True)


def pick_one_per_root(df: pd.DataFrame, id_col: str, root_col: str = "root_id") -> pd.DataFrame:
    """
    For roots with multiple slide-level IDs (a, b, c),
    pick the 'a' slide if present, otherwise the first row.
    """
    def _pick(group: pd.DataFrame) -> pd.Series:
        ids = group[id_col].astype(str)
        mask_a = ids.str.endswith("a")
        if mask_a.any():
            return group.loc[mask_a].iloc[0]
        return group.iloc[0]

    picked = df.groupby(root_col, group_keys=False).apply(_pick)
    return picked


# 1) MHC scores (root-level)
mhc_raw = pd.read_parquet(PATH_MHC)

mhc_raw["root_id"] = mhc_raw["Subject Id"].astype(str)
mhc = mhc_raw.set_index("root_id")[["burden_count", "weighted_burden", "top_n_avg_score"]].copy()
mhc = mhc.rename(
    columns={
        "burden_count": "MHC_burden_count",
        "weighted_burden": "MHC_weighted_burden",
        "top_n_avg_score": "MHC_top_n_avg_score",
    }
)

# 2) Mutation matrix (slide-level -> root-level, pick 'a')
mut_raw = pd.read_csv(PATH_MUT)
if "Unnamed: 0" not in mut_raw.columns:
    raise KeyError("Expected 'Unnamed: 0' column as sample ID in mutation matrix.")

mut_raw = mut_raw.rename(columns={"Unnamed: 0": "Original_ID"})
mut_raw["root_id"] = add_root_id(mut_raw["Original_ID"])

mut_picked = pick_one_per_root(mut_raw, id_col="Original_ID", root_col="root_id")

# drop the ID columns from the mutation feature block
mut_features = mut_picked.drop(columns=["Original_ID"])
# prefix all remaining columns with Mut_
mut_features = mut_features.rename(columns={c: f"Mut_{c}" for c in mut_features.columns})
mut_features.index.name = "root_id"

# 3) Clinical (slide-level -> root-level, pick 'a')
clin_cols = [
    PATIENT_ID_COL_CLIN,
    "MGMT promoter methylation",
    "age_at_diagnosis",
    "p53",
    "center",
    "Administrative gender",
]

clin_base = clin_out[clin_cols].copy()
clin_base = clin_base.rename(
    columns={
        PATIENT_ID_COL_CLIN: "Original_ID",
        "MGMT promoter methylation": "MGMT_status",
        "p53": "p53_status",
        "Administrative gender": "Administrative_gender",
    }
)
clin_base["root_id"] = add_root_id(clin_base["Original_ID"])

clin_picked = pick_one_per_root(clin_base, id_col="Original_ID", root_col="root_id")
clin_picked = clin_picked.set_index("root_id")

# MGMT binary
clin_picked["MGMT_binary"] = clin_picked["MGMT_status"].map({"Methylated": 1, "Unmethylated": 0})

# p53 binary
p53_clean = (
    clin_picked["p53_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"nan": np.nan})
)
clin_picked["p53_positive"] = p53_clean.map({"positive": 1, "negative": 0})

# 4) Define target roots: MHC ∩ mutation ∩ clinical
roots_mhc = set(mhc.index)
roots_mut = set(mut_features.index)
roots_clin = set(clin_picked.index)

target_roots = sorted(roots_mhc & roots_mut & roots_clin)
print(f"Number of target roots (MHC ∩ mutation ∩ clinical): {len(target_roots)}")

# Restrict core blocks to target roots
mhc_core = mhc.reindex(target_roots)
mut_core = mut_features.reindex(target_roots)
clin_core = clin_picked.reindex(target_roots)

# 5) Visium TIL (TIL1 / TIL2), per patient root (using 'a' slide where there are multiple)
key = adata.uns["celltype_annotation_key"]
lvl4_index = key.index.astype(str)
lvl2_vals = key[LEVEL_COL].astype(str)
tnk_level4 = lvl4_index[lvl2_vals.isin(TIL_GROUPS)]

c2l_cols = [
    f"meanscell_abundance_w_sf_{ct}"
    for ct in tnk_level4
    if f"meanscell_abundance_w_sf_{ct}" in adata.obs.columns
]
if len(c2l_cols) == 0:
    raise ValueError("No C2L columns for T_NK found in adata.obs.")

# TIL per spot, then per Visium slide sample
adata.obs["TIL_count"] = adata.obs[c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

til_by_sample = adata.obs.groupby("sample", observed=True)["TIL_positive"].agg(
    TIL2_spots_vis="sum",
    total_spots="count",
)
til_by_sample["TIL2_fraction_vis"] = til_by_sample["TIL2_spots_vis"] / til_by_sample["total_spots"]

til_vis = til_by_sample.reset_index().rename(columns={"sample": "sample_id"})
til_vis["sample_root"] = til_vis["sample_id"].str.replace("_vis$", "", regex=True)
til_vis["root_id"] = add_root_id(til_vis["sample_root"])

# pick 'a' slide per root, then compute TIL1 based on fixed cutoff
til_picked = pick_one_per_root(til_vis, id_col="sample_root", root_col="root_id")
til_picked = til_picked.set_index("root_id")

til_picked["TIL1_group_vis"] = np.where(
    til_picked["TIL2_fraction_vis"] >= TIL_Q75_CUTOFF, "TIL_high", "TIL_low"
)
til_picked["TIL1_binary_vis"] = (til_picked["TIL1_group_vis"] == "TIL_high").astype(float)

til_core = til_picked.reindex(target_roots)

# 6) scRNA L4: broad invasive module (TIL3) and tight module (TIL4) + L4 cell proportions

MODULE_BROAD_CT4 = [
    "Microglia_intermediate_reactive",
    "NK_nos",
    "cDC2",
    "pDC",
    "NK_KLRC1_pos",
    "Macrophage_C1Q_pos",
    "T_cell_IFN_response",
    "T_cell_CD4",
    "T_cell_CD8",
    "T_NK_cycling",
    "cDC1",
    "T_reg",
    "Mast_cell",
    "Plasma_cell",
    "Plasmablast",
    "Perivascular_fibroblast_cycling",
    "Perivascular_fibroblast_type-I",
    "Pericyte_fibro-like",
]

MODULE_TIGHT_CT4 = [
    "pDC",
    "NK_KLRC1_pos",
    "Macrophage_C1Q_pos",
    "Macrophage_IFN_response",
    "T_cell_IFN_response",
    "T_cell_CD4",
    "T_cell_CD8",
    "T_NK_cycling",
    "cDC1",
    "T_reg",
]

cell_L4_raw = pd.read_csv(PATH_L4)
if "sample_root" not in cell_L4_raw.columns:
    raise KeyError("Expected 'sample_root' column in level4 cell proportion file.")

cell_L4_raw["root_id"] = add_root_id(cell_L4_raw["sample_root"])

# Broad module (TIL3)
broad_cols = [c for c in MODULE_BROAD_CT4 if c in cell_L4_raw.columns]
print(f"Broad module L4 members found in L4 proportions: {len(broad_cols)}/{len(MODULE_BROAD_CT4)}")
cell_L4_raw["TIL3_broad_fraction_sc"] = cell_L4_raw[broad_cols].sum(axis=1)

# Tight module (TIL4)
tight_cols = [c for c in MODULE_TIGHT_CT4 if c in cell_L4_raw.columns]
print(f"Tight module L4 members found in L4 proportions: {len(tight_cols)}/{len(MODULE_TIGHT_CT4)}")
if len(tight_cols) == 0:
    raise ValueError("No tight-module L4 columns found; cannot build TIL4_tight_fraction_sc.")

cell_L4_raw["TIL4_tight_fraction_sc"] = cell_L4_raw[tight_cols].sum(axis=1)

# pick 'a' slide per root for L4
cell_L4_pick = pick_one_per_root(cell_L4_raw, id_col="sample_root", root_col="root_id")
cell_L4_pick = cell_L4_pick.set_index("root_id")

# TIL3, TIL4 cores
til3_core = cell_L4_pick[["TIL3_broad_fraction_sc"]].reindex(target_roots)
til4_core = cell_L4_pick[["TIL4_tight_fraction_sc"]].reindex(target_roots)

# L4 cell proportions (keep everything else)
cols_L4_keep = [
    c
    for c in cell_L4_pick.columns
    if c not in ["sample_root", "TIL3_broad_fraction_sc", "TIL4_tight_fraction_sc"]
]
cell_props_L4 = cell_L4_pick[cols_L4_keep].copy()
cell_props_L4 = cell_props_L4.rename(columns={c: f"L4_{c}" for c in cell_props_L4.columns})
cell_props_L4 = cell_props_L4.reindex(target_roots)

# 7) scRNA L3 cell proportions (pick 'a' slide)
cell_L3_raw = pd.read_csv(PATH_L3)
if "sample_root" not in cell_L3_raw.columns:
    raise KeyError("Expected 'sample_root' column in level3 cell proportion file.")

cell_L3_raw["root_id"] = add_root_id(cell_L3_raw["sample_root"])

cell_L3_pick = pick_one_per_root(cell_L3_raw, id_col="sample_root", root_col="root_id")
cell_L3_pick = cell_L3_pick.set_index("root_id")

cols_L3_keep = [c for c in cell_L3_pick.columns if c not in ["sample_root"]]
cell_props_L3 = cell_L3_pick[cols_L3_keep].copy()
cell_props_L3 = cell_props_L3.rename(columns={c: f"L3_{c}" for c in cell_props_L3.columns})
cell_props_L3 = cell_props_L3.reindex(target_roots)

# 8) GSEA pseudobulk scores (Tumor / Myeloid / Vascular / TNK), pick 'a' per root

def load_gsea_root_pref_a(path: str, prefix: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "patient_id" not in df.columns:
        raise KeyError(f"Expected 'patient_id' column in {path}")
    df["root_id"] = add_root_id(df["patient_id"])
    picked = pick_one_per_root(df, id_col="patient_id", root_col="root_id")
    picked = picked.set_index("root_id")
    picked = picked.drop(columns=["patient_id"])
    picked = picked.rename(columns={c: f"{prefix}_{c}" for c in picked.columns})
    return picked

gsea_tumor   = load_gsea_root_pref_a(PATH_GSEA_TUMOR,   "Tumor")
gsea_myeloid = load_gsea_root_pref_a(PATH_GSEA_MYELOID, "Myeloid")
gsea_vascular= load_gsea_root_pref_a(PATH_GSEA_VASC,    "Vascular")
gsea_tnk     = load_gsea_root_pref_a(PATH_GSEA_TNK,     "TNK")

gsea_tumor_core    = gsea_tumor.reindex(target_roots)
gsea_myeloid_core  = gsea_myeloid.reindex(target_roots)
gsea_vascular_core = gsea_vascular.reindex(target_roots)
gsea_tnk_core      = gsea_tnk.reindex(target_roots)

# 9) Assemble PLV table on target_roots (Option B: allow missing per modality)
dfs = [
    clin_core,
    til_core,
    til3_core,
    til4_core,
    cell_props_L3,
    cell_props_L4,
    gsea_tumor_core,
    gsea_myeloid_core,
    gsea_vascular_core,
    gsea_tnk_core,
    mut_core,
    mhc_core,
]

plv = pd.concat(dfs, axis=1, join="outer")
plv.index.name = "patient_id"

# Ensure Original_ID is present (from clinical block) and first
if "Original_ID" not in plv.columns:
    raise KeyError("Original_ID missing in assembled PLV table (clinical block).")

plv_out = plv.reset_index()

# 10) Sanity checks and save
print("Final PLV table shape (n_patients, n_features):", plv_out.shape)

key_cols = [
    "Original_ID",
    "TIL1_binary_vis",
    "TIL2_fraction_vis",
    "TIL3_broad_fraction_sc",
    "TIL4_tight_fraction_sc",
    "MGMT_binary",
    "Administrative_gender",
    "MHC_burden_count",
    "MHC_weighted_burden",
    "MHC_top_n_avg_score",
]
present_key_cols = [c for c in key_cols if c in plv_out.columns]

print("Non-missing counts for key columns:")
print(plv_out[present_key_cols].notna().sum())

plv_out.to_csv(OUT_CSV, index=False)
print(f"Saved PLV table to: {OUT_CSV}")
print(plv_out.head())

### TIL-high/low correlation analysis to patient level variables

In [ ]:
# %% TIL1–4 vs PLVs: unified association analysis (no manual feature filtering)

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, mannwhitneyu

PLV_PATH = "patient_level_variables_TIL_and_PLVs.csv"

# 1) Load PLV and standardize Administrative_gender
plv = pd.read_csv(PLV_PATH)
print(f"PLV table loaded. Shape: {plv.shape}")

if "Administrative_gender" in plv.columns:
    plv["Administrative_gender"] = (
        plv["Administrative_gender"]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan})
        .map({"Male": 1, "Female": 0})
    )


def run_til_associations(df, til_col, label, cont_out, bin_out):
    if til_col not in df.columns:
        raise KeyError(f"{til_col} not found in PLV table.")

    df = df.copy()
    df = df.dropna(subset=[til_col])
    til = df[til_col].astype(float)

    print(f"\n===== Associations for {til_col} ({label}) =====")
    print(f"Non-missing rows: {len(df)}")

    # 2) Decide which columns to treat as features (only exclude the TIL itself)
    feature_cols = [c for c in df.columns if c != til_col]

    continuous_features = []
    binary_features = []

    for col in feature_cols:
        s = df[col]
        non_na = s.dropna()
        if non_na.nunique() <= 1:
            continue

        if pd.api.types.is_bool_dtype(non_na):
            binary_features.append(col)
        elif pd.api.types.is_numeric_dtype(non_na):
            uniq = set(non_na.unique())
            if uniq.issubset({0, 1}) and len(uniq) == 2:
                binary_features.append(col)
            else:
                continuous_features.append(col)
        # non numeric are ignored

    print(
        f"Identified {len(continuous_features)} continuous PLVs and "
        f"{len(binary_features)} binary PLVs for association testing."
    )

    # 3) Continuous PLVs vs TIL: Spearman + Q4 vs Q1 Cohen's d
    cont_results = []

    q1 = til.quantile(0.25)
    q4 = til.quantile(0.75)

    for col in continuous_features:
        s = df[col].astype(float)
        mask_not_na = s.notna() & til.notna()
        s = s[mask_not_na]
        til_sub = til[mask_not_na]

        # require at least 5 paired samples
        if len(s) < 5 or til_sub.nunique() < 2:
            continue

        try:
            rho, p_rho = spearmanr(til_sub, s)
        except Exception:
            rho, p_rho = np.nan, np.nan

        # Q1 vs Q4 on TIL (works for binary TIL1 and continuous TIL2–4)
        s_q1 = s[til_sub <= q1]
        s_q4 = s[til_sub >= q4]

        mean_q1 = float(s_q1.mean()) if len(s_q1) > 0 else np.nan
        mean_q4 = float(s_q4.mean()) if len(s_q4) > 0 else np.nan
        delta = mean_q4 - mean_q1

        if len(s_q1) >= 5 and len(s_q4) >= 5:
            var_q1 = s_q1.var(ddof=1)
            var_q4 = s_q4.var(ddof=1)
            n_q1 = len(s_q1)
            n_q4 = len(s_q4)
            pooled_sd = np.sqrt(
                ((n_q1 - 1) * var_q1 + (n_q4 - 1) * var_q4) / (n_q1 + n_q4 - 2)
            )
            if pooled_sd > 0:
                cohen_d = delta / pooled_sd
            else:
                cohen_d = np.nan
        else:
            cohen_d = np.nan

        cont_results.append(
            {
                "feature": col,
                "n_non_na": int(len(s)),
                "rho_spearman": rho,
                "p_spearman": p_rho,
                "mean_feature_Q1low_TIL": mean_q1,
                "mean_feature_Q4high_TIL": mean_q4,
                "delta_Q4minusQ1_feature": delta,
                "cohen_d_Q4_vs_Q1_feature": cohen_d,
            }
        )

    df_cont = pd.DataFrame(cont_results)

    # 4) Binary PLVs vs TIL: Mann–Whitney on TIL distribution + Cohen's d
    bin_results = []

    for col in binary_features:
        s = df[col]
        mask_not_na = s.notna() & til.notna()
        s = s[mask_not_na]
        til_sub = til[mask_not_na]

        # at least 5 paired samples total
        if len(til_sub) < 5:
            continue

        if pd.api.types.is_bool_dtype(s):
            s_num = s.astype(int)
        else:
            s_num = s.astype(int)

        if s_num.nunique() < 2:
            continue

        til_0 = til_sub[s_num == 0]
        til_1 = til_sub[s_num == 1]

        # require at least 5 patients in each feature group
        if len(til_0) < 5 or len(til_1) < 5:
            continue

        mean_til0 = float(til_0.mean())
        mean_til1 = float(til_1.mean())
        delta_til = mean_til1 - mean_til0

        var0 = til_0.var(ddof=1)
        var1 = til_1.var(ddof=1)
        n0 = len(til_0)
        n1 = len(til_1)

        pooled_sd = np.sqrt(((n0 - 1) * var0 + (n1 - 1) * var1) / (n0 + n1 - 2))
        if pooled_sd > 0:
            d_til = delta_til / pooled_sd
        else:
            d_til = np.nan

        try:
            _, p_u = mannwhitneyu(til_0, til_1, alternative="two-sided")
        except ValueError:
            p_u = np.nan

        bin_results.append(
            {
                "feature": col,
                "n_non_na": int(len(til_sub)),
                "n_feat0": int((s_num == 0).sum()),
                "n_feat1": int((s_num == 1).sum()),
                "mean_TIL_feature0": mean_til0,
                "mean_TIL_feature1": mean_til1,
                "delta_TIL_feature1_minus_0": delta_til,
                "cohen_d_TIL_feature1_vs_0": d_til,
                "p_mannwhitney": p_u,
            }
        )

    df_bin = pd.DataFrame(bin_results)

    df_cont.to_csv(cont_out, index=False)
    df_bin.to_csv(bin_out, index=False)

    print(f"Saved continuous associations for {til_col} to: {cont_out}")
    print(f"Saved binary associations for {til_col} to: {bin_out}")

    return df_cont, df_bin


# 5) Run for TIL1–4 and write combined summary

til_configs = [
    ("TIL1_binary_vis",        "TIL1 (binary Visium TIL high vs low)",            "til1_vs_plv_continuous.csv",                 "til1_vs_plv_binary.csv"),
    ("TIL2_fraction_vis",      "TIL2 (Visium T_NK fraction)",                     "TIL2_fraction_vis_vs_plv_continuous.csv",    "TIL2_fraction_vis_vs_plv_binary.csv"),
    ("TIL3_broad_fraction_sc", "TIL3 (broad invasive immune perivascular module)","TIL3_broad_fraction_sc_vs_plv_continuous.csv","TIL3_broad_fraction_sc_vs_plv_binary.csv"),
    ("TIL4_tight_fraction_sc", "TIL4 (tight immune module)",                      "TIL4_tight_fraction_sc_vs_plv_continuous.csv","TIL4_tight_fraction_sc_vs_plv_binary.csv"),
]

all_cont = []
all_bin = []

for til_col, label, cont_out, bin_out in til_configs:
    df_cont, df_bin = run_til_associations(
        plv,
        til_col=til_col,
        label=label,
        cont_out=cont_out,
        bin_out=bin_out,
    )
    if not df_cont.empty:
        df_cont = df_cont.copy()
        df_cont["til_measure"] = til_col
        df_cont["feature_type"] = "continuous"
        all_cont.append(df_cont)

    if not df_bin.empty:
        df_bin = df_bin.copy()
        df_bin["til_measure"] = til_col
        df_bin["feature_type"] = "binary"
        all_bin.append(df_bin)

if all_cont or all_bin:
    combined = pd.concat(all_cont + all_bin, ignore_index=True)
    combined_out = "TIL1_4_vs_PLVs_all_associations.csv"
    combined.to_csv(combined_out, index=False)
    print(f"\nSaved combined associations for TIL1–4 to: {combined_out}")
else:
    print("\nNo associations produced for any TIL measure.")

In [ ]:
# %% TIL1_binary_vis vs PLVs (continuous + binary associations)

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr, fisher_exact

# 1) Load PLV table and restrict to patients with TIL1 info
plv_path = "patient_level_variables_TIL_and_PLVs.csv"
plv = pd.read_csv(plv_path)

TIL_COL = "TIL1_binary_vis"

if TIL_COL not in plv.columns:
    raise KeyError(f"{TIL_COL} not found in PLV table; available columns: {plv.columns.tolist()}")

df = plv.copy()

# Administrative_gender is string; convert to binary Male=1, Female=0 locally
if "Administrative_gender" in df.columns:
    df["Administrative_gender"] = (
        df["Administrative_gender"]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan})
        .map({"Male": 1, "Female": 0})
    )

# Ensure TIL1 is binary 0/1 and drop NAs
df[TIL_COL] = df[TIL_COL].astype(float)
df = df.dropna(subset=[TIL_COL])
df[TIL_COL] = df[TIL_COL].astype(int)

print(f"Rows with {TIL_COL} defined: {df.shape[0]}")
print(df[TIL_COL].value_counts().rename("TIL1_counts"))

# 2) Decide which columns to treat as "features"

# ID like and core TIL columns to exclude
exclude_cols = {
    TIL_COL,
    "TIL1_group_vis",
    "TIL2_spots_vis",
    "TIL2_fraction_vis",
    "TIL3_broad_fraction_sc",
    "total_spots",
    "patient_id",
    "Original_ID",
    "Tumour block ID for MOSAIC",
}

# Features to drop completely (optional explicit names)
EXCLUDE_FEATURES_BY_NAME = {
    "L3_Doublet",
}

# Prefix-based exclusions (all L4 Doublet cell types)
EXCLUDE_FEATURE_PREFIXES = [
    "L4_Doublet_",
]

feature_cols = []
for c in df.columns:
    if c in exclude_cols or c in EXCLUDE_FEATURES_BY_NAME:
        continue
    if any(c.startswith(pref) for pref in EXCLUDE_FEATURE_PREFIXES):
        continue
    feature_cols.append(c)

# 3) Split features into continuous vs binary
continuous_features = []
binary_features = []

for col in feature_cols:
    s = df[col]
    non_na = s.dropna()
    if non_na.nunique() <= 1:
        continue

    if pd.api.types.is_bool_dtype(non_na):
        binary_features.append(col)
    elif pd.api.types.is_numeric_dtype(non_na):
        uniq = set(non_na.unique())
        if uniq.issubset({0, 1}) and len(uniq) == 2:
            binary_features.append(col)
        else:
            continuous_features.append(col)
    # non numeric, non bool are ignored

print(f"Continuous PLVs to test: {len(continuous_features)}")
print(f"Binary PLVs to test: {len(binary_features)}")

# 4) Continuous PLVs: Mann–Whitney U + Spearman
cont_results = []

for col in continuous_features:
    s = df[col].astype(float)
    mask_not_na = s.notna()
    s = s[mask_not_na]
    til = df.loc[mask_not_na, TIL_COL]

    if til.nunique() < 2:
        continue

    s_high = s[til == 1]
    s_low = s[til == 0]

    if len(s_high) < 5 or len(s_low) < 5:
        continue

    try:
        stat_u, p_u = mannwhitneyu(s_high, s_low, alternative="two-sided")
    except ValueError:
        stat_u, p_u = np.nan, np.nan

    mean_high = float(s_high.mean())
    mean_low = float(s_low.mean())
    delta = mean_high - mean_low

    var_high = s_high.var(ddof=1)
    var_low = s_low.var(ddof=1)
    n_high = len(s_high)
    n_low = len(s_low)
    pooled_sd = np.sqrt(
        ((n_high - 1) * var_high + (n_low - 1) * var_low)
        / (n_high + n_low - 2)
    )
    if pooled_sd > 0:
        cohen_d = delta / pooled_sd
    else:
        cohen_d = np.nan

    try:
        rho, p_rho = spearmanr(s, til)
    except Exception:
        rho, p_rho = np.nan, np.nan

    cont_results.append(
        {
            "feature": col,
            "n_total": int(len(s)),
            "n_high": int((til == 1).sum()),
            "n_low": int((til == 0).sum()),
            "mean_high": mean_high,
            "mean_low": mean_low,
            "delta_high_minus_low": delta,
            "cohen_d": cohen_d,
            "spearman_rho": rho,
            "p_mannwhitney": p_u,
            "p_spearman": p_rho,
        }
    )

df_cont = pd.DataFrame(cont_results)

if not df_cont.empty:
    df_cont = df_cont.sort_values("p_mannwhitney", ascending=True)
    print("\nTop 15 continuous PLVs associated with TIL1_binary_vis (by Mann–Whitney p):")
    print(df_cont.head(15))
else:
    print("\nNo continuous PLVs passed basic QC for association testing.")

# 5) Binary PLVs: Fisher exact test and odds ratio
bin_results = []

for col in binary_features:
    s = df[col]
    mask_not_na = s.notna()
    s = s[mask_not_na]
    til = df.loc[mask_not_na, TIL_COL]

    if pd.api.types.is_bool_dtype(s):
        s_num = s.astype(int)
    else:
        s_num = s.astype(int)

    # need both feature=0 and feature=1 represented
    if s_num.nunique() < 2 or til.nunique() < 2:
        continue

    # require at least 5 patients in each feature group
    til_0 = til[s_num == 0]
    til_1 = til[s_num == 1]
    if len(til_0) < 5 or len(til_1) < 5:
        continue

    a = int(((til == 0) & (s_num == 0)).sum())
    b = int(((til == 0) & (s_num == 1)).sum())
    c = int(((til == 1) & (s_num == 0)).sum())
    d = int(((til == 1) & (s_num == 1)).sum())

    if (a + b + c + d) == 0:
        continue

    table = [[a, b], [c, d]]

    try:
        OR, p_fisher = fisher_exact(table, alternative="two-sided")
    except Exception:
        OR, p_fisher = np.nan, np.nan

    bin_results.append(
        {
            "feature": col,
            "n_total": int(len(s)),
            "TIL_low_feature0": a,
            "TIL_low_feature1": b,
            "TIL_high_feature0": c,
            "TIL_high_feature1": d,
            "odds_ratio_feature1_in_TIL_high_vs_low": OR,
            "p_fisher": p_fisher,
        }
    )

df_bin = pd.DataFrame(bin_results)

if not df_bin.empty:
    df_bin = df_bin.sort_values("p_fisher", ascending=True)
    print("\nTop 15 binary PLVs associated with TIL1_binary_vis (by Fisher p):")
    print(df_bin.head(15))
else:
    print("\nNo binary PLVs passed basic QC for association testing.")

# 6) Save results
cont_out = "til1_vs_plv_continuous.csv"
bin_out = "til1_vs_plv_binary.csv"

df_cont.to_csv(cont_out, index=False)
df_bin.to_csv(bin_out, index=False)

print(f"\nSaved continuous associations to: {cont_out}")
print(f"Saved binary associations to:     {bin_out}")

### Visualize TIL correlation analyses

In [ ]:
# %% Volcano + binary bar plots for TIL1–4 vs PLVs (with optional feature filtering)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text

plt.rcParams["axes.grid"] = False

# ---------- user parameters ----------
TIL_CONFIGS = [
    {
        "til_name": "TIL1_binary_vis",
        "label": "TIL1 (binary Visium TIL high vs low)",
        "cont_path": "til1_vs_plv_continuous.csv",
        "bin_path": "til1_vs_plv_binary.csv",
    },
    {
        "til_name": "TIL2_fraction_vis",
        "label": "TIL2 (Visium T_NK fraction)",
        "cont_path": "TIL2_fraction_vis_vs_plv_continuous.csv",
        "bin_path": "TIL2_fraction_vis_vs_plv_binary.csv",
    },
    {
        "til_name": "TIL3_broad_fraction_sc",
        "label": "TIL3 (broad invasive immune/perivascular module)",
        "cont_path": "TIL3_broad_fraction_sc_vs_plv_continuous.csv",
        "bin_path": "TIL3_broad_fraction_sc_vs_plv_binary.csv",
    },
    {
        "til_name": "TIL4_tight_fraction_sc",
        "label": "TIL4 (tight immune module)",
        "cont_path": "TIL4_tight_fraction_sc_vs_plv_continuous.csv",
        "bin_path": "TIL4_tight_fraction_sc_vs_plv_binary.csv",
    },
]

# plot-time feature filters
EXCLUDE_FEATURES_EXACT = set([
    # put exact names here if you want to hide them
    # e.g. "TIL1_binary_vis", "TIL2_fraction_vis"
])

EXCLUDE_FEATURES_CONTAINS = [
    "Doublet",  # hide all Doublet features by default
    "TIL2_spots_vis",
    "TIL2_fraction_vis",
    "TIL4_tight_fraction_sc",
    "L4_Macrophage_C1Q_pos",
    "L3_Macrophage",
    "L3_T_cell",
    "L3_B_cell",
    "TIL1_binary_vis",
    "L4_T_cell_CD8",
    "L4_T_cell_CD4",
    "TIL3_broad_fraction_sc",
    "L4_T_reg"
]

P_SIG = 0.01
ANNOTATE_TOP = 60
P_EPS = 1e-300

FIGSIZE_VOLCANO = (8, 5)
FIGSIZE_BIN = (6, 4)

FONTSIZE_LABEL = 8
FONTSIZE_TICK = 6
FONTSIZE_TITLE = 10
FONTSIZE_ANNOT = 6
POINT_SIZE = 14
# -------------------------------------


def feature_passes_plot_filter(name: str) -> bool:
    if name in EXCLUDE_FEATURES_EXACT:
        return False
    for sub in EXCLUDE_FEATURES_CONTAINS:
        if sub and sub in name:
            return False
    return True


for cfg in TIL_CONFIGS:
    til_name = cfg["til_name"]
    label = cfg["label"]
    cont_path = cfg["cont_path"]
    bin_path = cfg["bin_path"]

    print(f"\n=== Plotting {til_name} ({label}) ===")

    # ---------- continuous / volcano ----------
    try:
        df_cont = pd.read_csv(cont_path)
    except FileNotFoundError:
        print(f"  Continuous file not found, skipping: {cont_path}")
        continue

    if df_cont.empty or "p_spearman" not in df_cont.columns:
        print(f"  No continuous results for {til_name}, skipping volcano.")
    else:
        df_cont = df_cont.dropna(subset=["p_spearman", "rho_spearman"])
        if df_cont.empty:
            print(f"  All continuous rows have NaN p/rho for {til_name}, skipping volcano.")
        else:
            df_cont = df_cont.copy()
            df_cont = df_cont[df_cont["feature"].apply(feature_passes_plot_filter)]
            if df_cont.empty:
                print(f"  No continuous features left after filtering for {til_name}, skipping volcano.")
            else:
                df_cont["neglog10_p"] = -np.log10(df_cont["p_spearman"].clip(lower=P_EPS))
                df_cont["sign"] = np.sign(df_cont["rho_spearman"]).map(
                    {1.0: "positive", -1.0: "negative", 0.0: "zero"}
                )
                df_cont["is_sig"] = df_cont["p_spearman"] < P_SIG

                fig, ax = plt.subplots(figsize=FIGSIZE_VOLCANO)

                ax.scatter(
                    df_cont["rho_spearman"],
                    df_cont["neglog10_p"],
                    s=POINT_SIZE,
                    alpha=0.35,
                    color="0.7",
                    edgecolors="none",
                )

                sig_mask = df_cont["is_sig"]
                pos_mask = sig_mask & (df_cont["sign"] == "positive")
                neg_mask = sig_mask & (df_cont["sign"] == "negative")

                if pos_mask.any():
                    ax.scatter(
                        df_cont.loc[pos_mask, "rho_spearman"],
                        df_cont.loc[pos_mask, "neglog10_p"],
                        s=POINT_SIZE,
                        alpha=0.8,
                        color="tab:red",
                        edgecolors="none",
                        label=f"ρ > 0, p < {P_SIG:.2g}",
                    )
                if neg_mask.any():
                    ax.scatter(
                        df_cont.loc[neg_mask, "rho_spearman"],
                        df_cont.loc[neg_mask, "neglog10_p"],
                        s=POINT_SIZE,
                        alpha=0.8,
                        color="tab:blue",
                        edgecolors="none",
                        label=f"ρ < 0, p < {P_SIG:.2g}",
                    )

                y_ref = -np.log10(P_SIG)
                ax.axhline(y=y_ref, linestyle="--", linewidth=1, alpha=0.5)
                x0, x1 = ax.get_xlim()
                ax.text(
                    x0 + 0.01 * (x1 - x0),
                    y_ref + 0.1,
                    f"p = {P_SIG:g}",
                    fontsize=FONTSIZE_TICK,
                    ha="left",
                    va="bottom",
                )

                ax.axvline(0, color="black", linewidth=1)

                df_annot = (
                    df_cont[df_cont["is_sig"]]
                    .sort_values("p_spearman", ascending=True)
                    .head(ANNOTATE_TOP)
                )

                texts = []
                xs = []
                ys = []
                for _, row in df_annot.iterrows():
                    x = row["rho_spearman"]
                    y = row["neglog10_p"]
                    txt = ax.text(
                        x,
                        y,
                        row["feature"],
                        fontsize=FONTSIZE_ANNOT,
                        ha="left",
                        va="bottom",
                    )
                    texts.append(txt)
                    xs.append(x)
                    ys.append(y)

                if texts:
                    adjust_text(
                        texts,
                        xs,
                        ys,
                        ax=ax,
                        expand_points=(1.2, 1.4),
                        expand_text=(1.2, 1.4),
                        arrowprops=dict(arrowstyle="-", lw=0.4, alpha=0.6),
                        force_text=0.5,
                        force_points=0.2,
                        lim=500,
                    )

                ax.set_xlabel(f"Spearman ρ ({til_name} vs feature)", fontsize=FONTSIZE_LABEL)
                ax.set_ylabel(r"$-\log_{10}(p_{\mathrm{Spearman}})$", fontsize=FONTSIZE_LABEL)
                ax.set_title(f"{label} vs continuous PLVs", fontsize=FONTSIZE_TITLE)

                ax.tick_params(labelsize=FONTSIZE_TICK)
                if pos_mask.any() or neg_mask.any():
                    ax.legend(fontsize=FONTSIZE_TICK, frameon=False)

                plt.tight_layout()
                plt.show()

    # ---------- binary / bar plot ----------
    try:
        df_bin = pd.read_csv(bin_path)
    except FileNotFoundError:
        print(f"  Binary file not found, skipping: {bin_path}")
        continue

    if df_bin.empty or "p_mannwhitney" not in df_bin.columns:
        print(f"  No binary results for {til_name}, skipping bar plot.")
        continue

    df_bin = df_bin.dropna(subset=["p_mannwhitney", "cohen_d_TIL_feature1_vs_0"])
    if df_bin.empty:
        print(f"  All binary rows have NaN p/d for {til_name}, skipping bar plot.")
        continue

    df_bin_plot = df_bin.copy()
    df_bin_plot = df_bin_plot[df_bin_plot["feature"].apply(feature_passes_plot_filter)]
    if df_bin_plot.empty:
        print(f"  No binary features left after filtering for {til_name}, skipping bar plot.")
        continue

    df_bin_plot = df_bin_plot.sort_values(
        "cohen_d_TIL_feature1_vs_0",
        ascending=True,
    )

    fig, ax = plt.subplots(figsize=FIGSIZE_BIN)
    y_pos = np.arange(len(df_bin_plot))

    ax.barh(
        y_pos,
        df_bin_plot["cohen_d_TIL_feature1_vs_0"],
    )

    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_bin_plot["feature"], fontsize=FONTSIZE_TICK)
    ax.set_xlabel(f"Cohen's d ({til_name}, feature=1 vs 0)", fontsize=FONTSIZE_LABEL)
    ax.set_title(f"{label} vs binary PLVs", fontsize=FONTSIZE_TITLE)
    ax.tick_params(axis="x", labelsize=FONTSIZE_TICK)

    plt.tight_layout()
    plt.show()

### Check point-biserial correlation

In [ ]:
# %% TIL1 vs continuous PLVs: Cohen's d + point-biserial r sanity check

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from adjustText import adjust_text

# ---------- user-adjustable parameters ----------
PLV_PATH = "patient_level_variables_TIL_and_PLVs.csv"
TIL_COL = "TIL1_binary_vis"

P_EPS = 1e-300
P_THRESH = 0.05          # significance threshold for coloring / labels
ANNOTATE_TOP = 40        # max number of features to label on volcano

FIGSIZE_CONT = (14, 8)

FONTSIZE_LABEL = 14
FONTSIZE_TICK = 8
FONTSIZE_TITLE = 14
FONTSIZE_ANNOT = 8

OUT_CSV = "til1_pointbiserial_vs_cohen.csv"
# ------------------------------------------------

# 1) Load PLV and prepare TIL1 + basic feature set
plv = pd.read_csv(PLV_PATH)

if TIL_COL not in plv.columns:
    raise KeyError(f"{TIL_COL} not found in PLV table; columns: {plv.columns.tolist()}")

df = plv.copy()

# Administrative_gender to numeric so it can be classified as binary if needed
if "Administrative_gender" in df.columns:
    df["Administrative_gender"] = (
        df["Administrative_gender"]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan})
        .map({"Male": 1, "Female": 0})
    )

# Ensure TIL1 is binary 0/1 and drop NAs
df[TIL_COL] = df[TIL_COL].astype(float)
df = df.dropna(subset=[TIL_COL])
df[TIL_COL] = df[TIL_COL].astype(int)

print(f"Rows with {TIL_COL} defined: {df.shape[0]}")
print(df[TIL_COL].value_counts().rename("TIL1_counts"))

# ID-like and core TIL columns to exclude from feature list
exclude_cols = {
    TIL_COL,
    "TIL1_group_vis",
    "TIL2_spots_vis",
    "TIL2_fraction_vis",
    "TIL3_broad_fraction_sc",
    "TIL4_tight_fraction_sc",
    "total_spots",
    "patient_id",
    "Original_ID",
    "Tumour block ID for MOSAIC",
}

# Explicit feature-name exclusions
EXCLUDE_FEATURES_BY_NAME = {
    "L3_Doublet",
}

# Prefix-based exclusions (e.g. Doublet cell types)
EXCLUDE_FEATURE_PREFIXES = [
    "L4_Doublet_",
]

# 2) Identify continuous features (same logic as main TIL1 analysis)
feature_cols = []
for c in df.columns:
    if c in exclude_cols or c in EXCLUDE_FEATURES_BY_NAME:
        continue
    if any(c.startswith(pref) for pref in EXCLUDE_FEATURE_PREFIXES):
        continue
    feature_cols.append(c)

continuous_features = []
for col in feature_cols:
    s = df[col]
    non_na = s.dropna()
    if non_na.nunique() <= 1:
        continue
    if pd.api.types.is_bool_dtype(non_na):
        continue
    if pd.api.types.is_numeric_dtype(non_na):
        uniq = set(non_na.unique())
        if uniq.issubset({0, 1}) and len(uniq) == 2:
            continue
        continuous_features.append(col)

print(f"Continuous PLVs to test (for point-biserial): {len(continuous_features)}")

# 3) Compute Cohen's d and point-biserial r for each continuous feature
results = []
til_full = df[TIL_COL]

for col in continuous_features:
    s = df[col].astype(float)
    mask = s.notna() & til_full.notna()
    s = s[mask]
    til = til_full[mask]

    if til.nunique() < 2:
        continue

    n_low = int((til == 0).sum())
    n_high = int((til == 1).sum())
    if n_low < 5 or n_high < 5:
        continue

    s_low = s[til == 0]
    s_high = s[til == 1]

    mean_low = float(s_low.mean())
    mean_high = float(s_high.mean())
    delta = mean_high - mean_low

    var_low = s_low.var(ddof=1)
    var_high = s_high.var(ddof=1)

    if (n_low + n_high - 2) > 0:
        pooled_sd = np.sqrt(
            ((n_low - 1) * var_low + (n_high - 1) * var_high) / (n_low + n_high - 2)
        )
    else:
        pooled_sd = np.nan

    if pooled_sd > 0:
        d = delta / pooled_sd
    else:
        d = np.nan

    try:
        r_pb, p_pb = pearsonr(til.values.astype(float), s.values.astype(float))
    except Exception:
        r_pb, p_pb = np.nan, np.nan

    results.append(
        {
            "feature": col,
            "n_total": int(len(s)),
            "n_low": n_low,
            "n_high": n_high,
            "mean_low": mean_low,
            "mean_high": mean_high,
            "delta_high_minus_low": delta,
            "cohen_d": d,
            "r_pointbiserial": r_pb,
            "p_pointbiserial": p_pb,
        }
    )

df_pb = pd.DataFrame(results)

if df_pb.empty:
    raise RuntimeError("No continuous features passed QC for point-biserial calculation.")

# 4) Prepare volcano data
df_pb["neglog10_p"] = -np.log10(df_pb["p_pointbiserial"].clip(lower=P_EPS))
df_pb["sign"] = np.sign(df_pb["r_pointbiserial"]).map(
    {-1.0: "negative", 0.0: "zero", 1.0: "positive"}
)
df_pb["is_sig"] = df_pb["p_pointbiserial"] < P_THRESH

df_pb_sorted = df_pb.sort_values("p_pointbiserial", ascending=True)

print("\nTop 15 features by point-biserial p-value:")
print(
    df_pb_sorted[
        [
            "feature",
            "cohen_d",
            "r_pointbiserial",
            "p_pointbiserial",
            "n_low",
            "n_high",
        ]
    ].head(15)
)

valid = df_pb[["cohen_d", "r_pointbiserial"]].dropna()
if len(valid) > 2:
    corr_dr = np.corrcoef(valid["cohen_d"], valid["r_pointbiserial"])[0, 1]
    print(f"\nCorrelation between Cohen's d and r_pointbiserial across features: {corr_dr:.3f}")

# 5) Volcano plot (x = r_pointbiserial, y = -log10 p)
plt.figure(figsize=FIGSIZE_CONT)
ax = plt.gca()
ax.grid(False)

ax.scatter(
    df_pb["r_pointbiserial"],
    df_pb["neglog10_p"],
    s=20,
    alpha=0.4,
    color="grey",
    linewidths=0,
    label=None,
)

for sign, color, label in [
    ("positive", "tab:red", f"r > 0, p < {P_THRESH:g}"),
    ("negative", "tab:blue", f"r < 0, p < {P_THRESH:g}"),
]:
    mask = (df_pb["sign"] == sign) & df_pb["is_sig"]
    if not mask.any():
        continue
    ax.scatter(
        df_pb.loc[mask, "r_pointbiserial"],
        df_pb.loc[mask, "neglog10_p"],
        s=30,
        alpha=0.9,
        color=color,
        linewidths=0.5,
        edgecolors="black",
        label=label,
    )

y_ref = -np.log10(P_THRESH)
ax.axhline(y=y_ref, linestyle="--", linewidth=1, alpha=0.6, color="black")
ax.text(
    0.02,
    y_ref + 0.1,
    f"p = {P_THRESH}",
    fontsize=FONTSIZE_TICK,
    va="bottom",
    ha="left",
)

ax.axvline(x=0.0, linestyle="-", linewidth=1, alpha=0.8, color="black")

df_annot = df_pb[df_pb["is_sig"]].sort_values("p_pointbiserial", ascending=True).head(ANNOTATE_TOP)

texts = []
xs = []
ys = []
for _, row in df_annot.iterrows():
    x = row["r_pointbiserial"]
    y = row["neglog10_p"]
    txt = ax.text(
        x,
        y,
        row["feature"],
        fontsize=FONTSIZE_ANNOT,
        ha="center",
        va="bottom",
    )
    texts.append(txt)
    xs.append(x)
    ys.append(y)

if texts:
    adjust_text(
        texts,
        xs,
        ys,
        ax=ax,
        arrowprops=dict(arrowstyle="-", lw=0.5, alpha=0.6),
        expand_points=(1.1, 1.1),
        expand_text=(1.1, 1.1),
        force_text=0.5,
        force_points=0.2,
        lim=500,
    )

ax.set_xlabel("Point-biserial r (TIL1_high vs TIL1_low)", fontsize=FONTSIZE_LABEL)
ax.set_ylabel("-log10(p_pointbiserial)", fontsize=FONTSIZE_LABEL)
ax.set_title("TIL1_binary_vis vs continuous PLVs (point-biserial correlation)", fontsize=FONTSIZE_TITLE)

if df_pb["is_sig"].any():
    ax.legend(fontsize=FONTSIZE_TICK, frameon=False)

ax.tick_params(axis="both", labelsize=FONTSIZE_TICK)
plt.tight_layout()
plt.show()

# 6) Save full table
df_pb_sorted.to_csv(OUT_CSV, index=False)
print(f"\nSaved detailed results (Cohen's d + point-biserial) to: {OUT_CSV}")

### Is MHC score significant?

In [ ]:
# %% Sanity check: were MHC_* variables actually included in TIL1/2/3 analyses?

import pandas as pd
import numpy as np

plv_path = "patient_level_variables_TIL_and_PLVs.csv"
plv = pd.read_csv(plv_path)
print(f"PLV shape: {plv.shape}")

# 1) Identify MHC columns in PLV
mhc_cols = [c for c in plv.columns if c.startswith("MHC_")]
print("\nMHC columns in PLV table:")
print(mhc_cols)

if not mhc_cols:
    raise ValueError("No MHC_* columns found in PLV table, something is wrong upstream.")

print("\nNon-missing counts for MHC columns (overall):")
print(plv[mhc_cols].notna().sum())

# 2) For each TIL measure, check:
#    - how many rows have that TIL defined
#    - how many of those rows have non-missing MHC
#    - whether the association CSVs contain MHC_* features

til_specs = {
    "TIL1_binary_vis": {
        "cont": "til1_vs_plv_continuous.csv",
        "bin":  "til1_vs_plv_binary.csv",
    },
    "TIL2_fraction_vis": {
        "cont": "TIL2_fraction_vis_vs_plv_continuous.csv",
        "bin":  "TIL2_fraction_vis_vs_plv_binary.csv",
    },
    "TIL3_broad_fraction_sc": {
        "cont": "TIL3_broad_fraction_sc_vs_plv_continuous.csv",
        "bin":  "TIL3_broad_fraction_sc_vs_plv_binary.csv",
    },
}

for til_col, paths in til_specs.items():
    print("\n" + "=" * 80)
    print(f"TIL measure: {til_col}")

    if til_col not in plv.columns:
        print(f"  {til_col} not found in PLV, skipping.")
        continue

    # Subset to rows with this TIL defined
    sub = plv.dropna(subset=[til_col])
    print(f"  Rows with {til_col} defined: {sub.shape[0]}")

    # Non-missing MHC counts in that subset
    print("  Non-missing MHC counts in this subset:")
    print(sub[mhc_cols].notna().sum())

    # Continuous results
    cont_path = paths["cont"]
    try:
        df_cont = pd.read_csv(cont_path)
        mhc_cont = df_cont[df_cont["feature"].isin(mhc_cols)]
        print(f"  MHC features present in {cont_path}: {mhc_cont.shape[0]}")
        if not mhc_cont.empty:
            # Sort by whichever p column exists
            p_col = "p_mannwhitney" if "p_mannwhitney" in mhc_cont.columns else "p_spearman"
            mhc_cont = mhc_cont.sort_values(p_col, ascending=True)
            print("  MHC rows in continuous results (head):")
            print(mhc_cont.head())
    except FileNotFoundError:
        print(f"  Continuous results file not found: {cont_path}")

    # Binary results (MHC are continuous so this should normally be zero)
    bin_path = paths["bin"]
    try:
        df_bin = pd.read_csv(bin_path)
        mhc_bin = df_bin[df_bin["feature"].isin(mhc_cols)]
        print(f"  MHC features present in {bin_path}: {mhc_bin.shape[0]}")
        if not mhc_bin.empty:
            print("  MHC rows in binary results (head):")
            print(mhc_bin.head())
    except FileNotFoundError:
        print(f"  Binary results file not found: {bin_path}")

### Clonal heterogeneity variable

In [ ]:
# %% ITH histogram + KM survival (using clin_out as DataFrame or path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

ITH_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/scRNAseq/infercnv/ith_scores.csv"
SURV_TIME_COL = "os_years"
SURV_EVENT_COL = "os_event"
ITH_METRIC = "ithcna"  # or "ithgex" if you prefer

# 1) Load raw ITH at cell level and drop rows with missing ITH
ith_raw = pd.read_csv(ITH_PATH, index_col=0)
print("Raw ITH table shape before dropna:", ith_raw.shape)
ith_raw = ith_raw.dropna(how="any")
print("Raw ITH table shape after dropna:", ith_raw.shape)

if ITH_METRIC not in ith_raw.columns:
    raise KeyError(f"{ITH_METRIC} not found in ITH table. Columns: {ith_raw.columns.tolist()}")

# 2) Derive patient_id from cell barcodes
# Example index: "CH_G_454a.AAACCAATCACTTGATAGCTGTGA-1"
#   -> take part before ".", then strip trailing letter from the numeric block
#      "CH_G_454a" -> "CH_G_454"
ith_raw = ith_raw.copy()
ith_raw["patient_id"] = (
    ith_raw.index.to_series()
    .str.split(".")
    .str[0]
    .str.replace(r"([0-9]+)[a-zA-Z]$", r"\1", regex=True)
)

print("\nExample mapping index -> patient_id:")
print(
    ith_raw[["patient_id"]]
    .head(10)
)

# 3) Patient level ITH: mean across cells per patient
ith_patient = (
    ith_raw
    .groupby("patient_id")[[ITH_METRIC]]
    .mean()
)

print("\nPatient-level ITH table:")
print("  shape:", ith_patient.shape)
print("  index examples:", ith_patient.index[:10].tolist())
print("  basic stats:")
print(ith_patient[ITH_METRIC].describe())

# 4) Histogram of patient-level ITH
plt.figure(figsize=(6, 4))
plt.hist(ith_patient[ITH_METRIC], bins=30, edgecolor="black", alpha=0.7)
plt.xlabel(ITH_METRIC)
plt.ylabel("Number of patients")
plt.title(f"Patient-level {ITH_METRIC} distribution")
plt.tight_layout()
plt.show()

# 5) Load clinical data and derive patient_id
#    clin_out can be either a DataFrame (already loaded) or a path string
if isinstance(clin_out, pd.DataFrame):
    clin = clin_out.copy()
    print("\nclin_out provided as DataFrame.")
else:
    print("\nclin_out provided as path or buffer, reading with read_csv.")
    clin = pd.read_csv(clin_out)

print("\nclin_out shape:", clin.shape)
print("clin_out columns (head):", clin.columns[:15].tolist())

if SURV_TIME_COL not in clin.columns or SURV_EVENT_COL not in clin.columns:
    raise KeyError(
        f"Survival columns '{SURV_TIME_COL}' or '{SURV_EVENT_COL}' not found in clin_out. "
        f"Columns: {clin.columns.tolist()}"
    )

clin = clin.copy()

# Derive patient_id for survival merge
if "root_id" in clin.columns:
    clin["patient_id"] = clin["root_id"].astype(str)
    print("Derived patient_id from clin['root_id'].")
elif "Tumour block ID for MOSAIC" in clin.columns:
    clin["patient_id"] = (
        clin["Tumour block ID for MOSAIC"]
        .astype(str)
        .str.replace(r"[a-zA-Z]$", "", regex=True)
    )
    print("Derived patient_id from clin['Tumour block ID for MOSAIC'] (dropping letter).")
elif "sample" in clin.columns:
    clin["patient_id"] = (
        clin["sample"]
        .astype(str)
        .str.replace("_vis", "", regex=False)
        .str.replace(r"[a-zA-Z]$", "", regex=True)
    )
    print("Derived patient_id from clin['sample'] (dropping '_vis' and trailing letter).")
else:
    raise KeyError(
        "Could not find a column in clin_out to derive patient_id "
        "(expected one of 'root_id', 'Tumour block ID for MOSAIC', or 'sample')."
    )

print("\nExample clinical patient_id mapping:")
print(
    clin[["patient_id", SURV_TIME_COL, SURV_EVENT_COL]]
    .head(10)
)

# 6) Build patient-level survival table
surv = (
    clin[["patient_id", SURV_TIME_COL, SURV_EVENT_COL]]
    .dropna(subset=[SURV_TIME_COL, SURV_EVENT_COL])
    .drop_duplicates(subset=["patient_id"])
    .set_index("patient_id")
)

print("\nSurvival table at patient level:")
print("  shape:", surv.shape)
print("  example rows:")
print(surv.head())

# 7) Merge ITH with survival on patient_id
merged = ith_patient.join(surv, how="inner")
print("\nMerge ITH + survival:")
print("  shape:", merged.shape)
print("  columns:", merged.columns.tolist())
print(merged.head())

if merged.shape[0] < 10:
    print("Warning: very few patients have both ITH and survival.")

# 8) Define ITH high vs rest using top quartile of ITH_METRIC
q75 = merged[ITH_METRIC].quantile(0.75)
merged["ITH_group"] = np.where(
    merged[ITH_METRIC] >= q75,
    "High_Q4",
    "Q1_Q3",
)

print("\nITH grouping:")
print("  Q4 cutoff:", q75)
print(merged["ITH_group"].value_counts())

# 9) KM survival curves and log-rank test
kmf = KaplanMeierFitter()

plt.figure(figsize=(6, 4))

groups = ["High_Q4", "Q1_Q3"]
colors = ["tab:red", "tab:blue"]

for group, color in zip(groups, colors):
    mask = merged["ITH_group"] == group
    if mask.sum() < 5:
        print(f"Group {group} has < 5 patients; KM curve may be unstable.")
    kmf.fit(
        merged.loc[mask, SURV_TIME_COL],
        merged.loc[mask, SURV_EVENT_COL],
        label=group,
    )
    kmf.plot(ci_show=False, color=color)

plt.xlabel("Overall survival (years)")
plt.ylabel("Survival probability")
plt.title(f"OS by {ITH_METRIC} (top quartile vs Q1–Q3)")
plt.tight_layout()
plt.show()

# Log-rank test
mask_high = merged["ITH_group"] == "High_Q4"
mask_rest = merged["ITH_group"] == "Q1_Q3"

res = logrank_test(
    merged.loc[mask_high, SURV_TIME_COL],
    merged.loc[mask_rest, SURV_TIME_COL],
    event_observed_A=merged.loc[mask_high, SURV_EVENT_COL],
    event_observed_B=merged.loc[mask_rest, SURV_EVENT_COL],
)

print("\nLog-rank test High_Q4 vs Q1_Q3:")
print(f"  p-value: {res.p_value:.3e}")
print(f"  test statistic: {res.test_statistic:.3f}")

In [ ]:
# %% Count True / 1 values per mutation column from file

import pandas as pd

BASE = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd"
PATH_MUT = f"{BASE}/results/WES/mutations/patient_gene_matrix.csv"

mut = pd.read_csv(PATH_MUT)

print("Mutation matrix shape:", mut.shape)

for col in mut.columns:
    # Skip obvious ID-like columns; keep ONLY the actual mutation flags
    if col in {"Unnamed: 0", "Original_ID", "root_id"}:
        continue

    s = mut[col]

    if pd.api.types.is_bool_dtype(s):
        n_true = int(s.sum())
    else:
        # Treat nonzero as "True" (works for 0/1 or counts)
        n_true = int((s.fillna(0) != 0).sum())

    print(f"{col}: {n_true}")